# Lab 02 — Predict, measure, explain an inference bottleneck

**Outcome:** follow one exact prompt through token IDs, actual generated output and KV state; reconstruct every decoder-block operation in a tiny CPU example and inspect recorded real activations; then produce a reproducible phase experiment, a matched batch comparison and two focused traces. Explain one prediction that did not survive measurement. This is not an HTTP benchmark or a quality test.

Read [Book 02](https://levshaazz.github.io/llm-serving-mastery/en/book/02/) and the [student guide](https://levshaazz.github.io/llm-serving-mastery/en/start/). You independently arrange a CUDA GPU (Linux/Colab/WSL; 16 GiB recommended). CPU contract checks do not replace the GPU run. Use your private repository's GitHub Issue with @levshaazz for diagnostics; do not post secrets or private artifacts publicly.

**Protocol v2:** two pinned models × five cases × five raw trials = **50 trials**. Cases are batch 1 × input {128,4096} × output {32,128}, plus batch 4/input 128/output 32. Batch 1/input 128/output 32 is the matched batch control. Output 512 and batch 8 are optional follow-ups only, after the core passes. Historical v1 results are not a v2 baseline.

**Before execution:** write three predictions: which phase changes with prompt length, which with output length, and whether batch 4 raises aggregate rate while changing per-sequence cadence. Name the measurement that could disprove each prediction.

Run one notebook/process at a time. Models reuse a persistent cache; execution artifacts get new directories. Stop on OOM, resource growth or a gate failure; preserve the partial JSON. Do not restart the entire matrix blindly.


## 0. See the exact prompt first — recorded replay, CPU only

Run the next two cells without installing anything or connecting a GPU. The actual recorded input and generated sequence are embedded in this notebook; they require only Python's standard library, no network request and no model download. This is **recorded instructor evidence**, not a GPU run you performed and not completion of the required lab.

**Predict before revealing:** the user asks “Continue the sequence with three numbers: 2, 4, 6,”. What else reaches the model? Does the first forward process one token or the entire prompt? After selecting its first output, has that new token already acquired K/V?

The natural-text replay uses a pinned 1.5B model and a maximum of 24 greedy output tokens, stopping at EOS. It synchronizes each step for inspection. The later required matrix uses synthetic repeated IDs, fixed output lengths and whole-phase timing. **Never attach the replay's natural prompt to the matrix's synthetic timing numbers.**


In [ ]:
import json
RECORDED_PROMPT = json.loads("{\"protocol\":\"topic02-concrete-prompt-v1\",\"scope\":\"one instrumented natural-EOS request; not a benchmark\",\"model\":{\"id\":\"Qwen/Qwen2.5-1.5B-Instruct\",\"revision\":\"989aa7980e4cf806f80c7fef2b1adb7bc71aa306\",\"kv_bytes_per_token\":28672,\"layers\":28,\"kv_heads\":2,\"attention_heads\":12,\"hidden_size\":1536,\"attention_selected\":\"sdpa\"},\"environment\":{\"torch\":\"2.14.0+cu130\",\"transformers\":\"4.51.3\",\"cuda\":\"13.0\",\"gpu\":\"NVIDIA GeForce RTX 5070 Ti\",\"dtype\":\"float16\",\"attention_requested\":\"sdpa\",\"logits_to_keep\":1},\"generation\":{\"max_new_tokens\":24,\"do_sample\":false,\"batch\":1,\"eos_token_ids\":[151645,151643]},\"measurement\":{\"warmup_requests\":1,\"measured_requests\":1,\"interval\":\"synchronized wall and CUDA events: forward + argmax; excludes CPU token extraction/text rendering/resource checks\",\"memory_peak_scope\":\"reset before each individual forward+argmax; model remains loaded\",\"limitations\":[\"single request, no distribution or performance guarantee\",\"per-token synchronization and inspection alter the execution\",\"not HTTP TTFT/TPOT, not summed CUDA kernel time, not protocol v2\"]},\"resource_samples\":[{\"stage\":\"before_load\",\"host_available_bytes\":74161643520,\"device_used_bytes\":1356398592,\"allocated_bytes\":0,\"reserved_bytes\":0,\"peak_allocated_bytes\":0,\"peak_reserved_bytes\":0},{\"stage\":\"post_load\",\"host_available_bytes\":71310524416,\"device_used_bytes\":4642635776,\"allocated_bytes\":3088347136,\"reserved_bytes\":3286237184,\"peak_allocated_bytes\":3088347136,\"peak_reserved_bytes\":3286237184},{\"stage\":\"after_warmup\",\"host_available_bytes\":71274737664,\"device_used_bytes\":4737007616,\"allocated_bytes\":3121901568,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3125901824,\"peak_reserved_bytes\":3323985920},{\"stage\":\"after_replay\",\"host_available_bytes\":71274737664,\"device_used_bytes\":4737007616,\"allocated_bytes\":3123364864,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920},{\"stage\":\"after_release\",\"host_available_bytes\":71274799104,\"device_used_bytes\":1446576128,\"allocated_bytes\":33554432,\"reserved_bytes\":33554432,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920}],\"prompt\":{\"messages\":[{\"role\":\"system\",\"content\":\"You are a helpful assistant. Reply with only the requested numbers.\"},{\"role\":\"user\",\"content\":\"Continue the sequence with three numbers: 2, 4, 6,\"}],\"rendered_chat\":\"<|im_start|>system\\nYou are a helpful assistant. Reply with only the requested numbers.<|im_end|>\\n<|im_start|>user\\nContinue the sequence with three numbers: 2, 4, 6,<|im_end|>\\n<|im_start|>assistant\\n\",\"input_token_ids\":[151644,8948,198,2610,525,264,10950,17847,13,17841,448,1172,279,11223,5109,13,151645,198,151644,872,198,23526,279,8500,448,2326,5109,25,220,17,11,220,19,11,220,21,11,151645,198,151644,77091,198],\"input_token_count\":42,\"input_tokens\":[{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":0},{\"id\":8948,\"piece\":\"system\",\"text\":\"system\",\"index\":1},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":2},{\"id\":2610,\"piece\":\"You\",\"text\":\"You\",\"index\":3},{\"id\":525,\"piece\":\"Ġare\",\"text\":\" are\",\"index\":4},{\"id\":264,\"piece\":\"Ġa\",\"text\":\" a\",\"index\":5},{\"id\":10950,\"piece\":\"Ġhelpful\",\"text\":\" helpful\",\"index\":6},{\"id\":17847,\"piece\":\"Ġassistant\",\"text\":\" assistant\",\"index\":7},{\"id\":13,\"piece\":\".\",\"text\":\".\",\"index\":8},{\"id\":17841,\"piece\":\"ĠReply\",\"text\":\" Reply\",\"index\":9},{\"id\":448,\"piece\":\"Ġwith\",\"text\":\" with\",\"index\":10},{\"id\":1172,\"piece\":\"Ġonly\",\"text\":\" only\",\"index\":11},{\"id\":279,\"piece\":\"Ġthe\",\"text\":\" the\",\"index\":12},{\"id\":11223,\"piece\":\"Ġrequested\",\"text\":\" requested\",\"index\":13},{\"id\":5109,\"piece\":\"Ġnumbers\",\"text\":\" numbers\",\"index\":14},{\"id\":13,\"piece\":\".\",\"text\":\".\",\"index\":15},{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\",\"index\":16},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":17},{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":18},{\"id\":872,\"piece\":\"user\",\"text\":\"user\",\"index\":19},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":20},{\"id\":23526,\"piece\":\"Continue\",\"text\":\"Continue\",\"index\":21},{\"id\":279,\"piece\":\"Ġthe\",\"text\":\" the\",\"index\":22},{\"id\":8500,\"piece\":\"Ġsequence\",\"text\":\" sequence\",\"index\":23},{\"id\":448,\"piece\":\"Ġwith\",\"text\":\" with\",\"index\":24},{\"id\":2326,\"piece\":\"Ġthree\",\"text\":\" three\",\"index\":25},{\"id\":5109,\"piece\":\"Ġnumbers\",\"text\":\" numbers\",\"index\":26},{\"id\":25,\"piece\":\":\",\"text\":\":\",\"index\":27},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":28},{\"id\":17,\"piece\":\"2\",\"text\":\"2\",\"index\":29},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":30},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":31},{\"id\":19,\"piece\":\"4\",\"text\":\"4\",\"index\":32},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":33},{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \",\"index\":34},{\"id\":21,\"piece\":\"6\",\"text\":\"6\",\"index\":35},{\"id\":11,\"piece\":\",\",\"text\":\",\",\"index\":36},{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\",\"index\":37},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":38},{\"id\":151644,\"piece\":\"<|im_start|>\",\"text\":\"<|im_start|>\",\"index\":39},{\"id\":77091,\"piece\":\"assistant\",\"text\":\"assistant\",\"index\":40},{\"id\":198,\"piece\":\"Ċ\",\"text\":\"\\n\",\"index\":41}]},\"replay\":{\"output_token_ids\":[23,11,220,16,15,11,220,16,17,151645],\"output_text\":\"8, 10, 12\",\"output_text_with_special_tokens\":\"8, 10, 12<|im_end|>\",\"stop_reason\":\"eos\",\"steps\":[{\"output_index\":1,\"phase\":\"prefill\",\"input_shape\":[1,42],\"input_token_ids\":[151644,8948,198,2610,525,264,10950,17847,13,17841,448,1172,279,11223,5109,13,151645,198,151644,872,198,23526,279,8500,448,2326,5109,25,220,17,11,220,19,11,220,21,11,151645,198,151644,77091,198],\"kv_tokens_before\":0,\"kv_tokens_after\":42,\"selected_token\":{\"id\":23,\"piece\":\"8\",\"text\":\"8\"},\"decoded_prefix\":\"8\",\"wall_ms\":19.63335300001745,\"cuda_event_ms\":19.085664749145508,\"memory\":{\"allocated_bytes\":3123106304,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3125901824,\"peak_reserved_bytes\":3323985920}},{\"output_index\":2,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[23],\"kv_tokens_before\":42,\"kv_tokens_after\":43,\"selected_token\":{\"id\":11,\"piece\":\",\",\"text\":\",\"},\"decoded_prefix\":\"8,\",\"wall_ms\":81.22387200000958,\"cuda_event_ms\":81.15577697753906,\"memory\":{\"allocated_bytes\":3123135488,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123442176,\"peak_reserved_bytes\":3323985920}},{\"output_index\":3,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[11],\"kv_tokens_before\":43,\"kv_tokens_after\":44,\"selected_token\":{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \"},\"decoded_prefix\":\"8, \",\"wall_ms\":28.601944000001822,\"cuda_event_ms\":28.543039321899414,\"memory\":{\"allocated_bytes\":3123164160,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123470848,\"peak_reserved_bytes\":3323985920}},{\"output_index\":4,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[220],\"kv_tokens_before\":44,\"kv_tokens_after\":45,\"selected_token\":{\"id\":16,\"piece\":\"1\",\"text\":\"1\"},\"decoded_prefix\":\"8, 1\",\"wall_ms\":20.453992999819093,\"cuda_event_ms\":20.372831344604492,\"memory\":{\"allocated_bytes\":3123192832,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123499520,\"peak_reserved_bytes\":3323985920}},{\"output_index\":5,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[16],\"kv_tokens_before\":45,\"kv_tokens_after\":46,\"selected_token\":{\"id\":15,\"piece\":\"0\",\"text\":\"0\"},\"decoded_prefix\":\"8, 10\",\"wall_ms\":33.23044800004027,\"cuda_event_ms\":33.17951965332031,\"memory\":{\"allocated_bytes\":3123221504,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123528192,\"peak_reserved_bytes\":3323985920}},{\"output_index\":6,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[15],\"kv_tokens_before\":46,\"kv_tokens_after\":47,\"selected_token\":{\"id\":11,\"piece\":\",\",\"text\":\",\"},\"decoded_prefix\":\"8, 10,\",\"wall_ms\":16.885007999917434,\"cuda_event_ms\":16.802719116210938,\"memory\":{\"allocated_bytes\":3123250176,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123556864,\"peak_reserved_bytes\":3323985920}},{\"output_index\":7,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[11],\"kv_tokens_before\":47,\"kv_tokens_after\":48,\"selected_token\":{\"id\":220,\"piece\":\"Ġ\",\"text\":\" \"},\"decoded_prefix\":\"8, 10, \",\"wall_ms\":68.2125509999878,\"cuda_event_ms\":68.14598083496094,\"memory\":{\"allocated_bytes\":3123278848,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123591680,\"peak_reserved_bytes\":3323985920}},{\"output_index\":8,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[220],\"kv_tokens_before\":48,\"kv_tokens_after\":49,\"selected_token\":{\"id\":16,\"piece\":\"1\",\"text\":\"1\"},\"decoded_prefix\":\"8, 10, 1\",\"wall_ms\":17.700886000056926,\"cuda_event_ms\":17.647199630737305,\"memory\":{\"allocated_bytes\":3123307520,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123626496,\"peak_reserved_bytes\":3323985920}},{\"output_index\":9,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[16],\"kv_tokens_before\":49,\"kv_tokens_after\":50,\"selected_token\":{\"id\":17,\"piece\":\"2\",\"text\":\"2\"},\"decoded_prefix\":\"8, 10, 12\",\"wall_ms\":55.26893700016444,\"cuda_event_ms\":55.212703704833984,\"memory\":{\"allocated_bytes\":3123336192,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123661312,\"peak_reserved_bytes\":3323985920}},{\"output_index\":10,\"phase\":\"decode\",\"input_shape\":[1,1],\"input_token_ids\":[17],\"kv_tokens_before\":50,\"kv_tokens_after\":51,\"selected_token\":{\"id\":151645,\"piece\":\"<|im_end|>\",\"text\":\"<|im_end|>\"},\"decoded_prefix\":\"8, 10, 12<|im_end|>\",\"wall_ms\":16.745422999974835,\"cuda_event_ms\":16.682655334472656,\"memory\":{\"allocated_bytes\":3123364864,\"reserved_bytes\":3323985920,\"peak_allocated_bytes\":3123696128,\"peak_reserved_bytes\":3323985920}}],\"final_kv_tokens\":51,\"instrumented_request_wall_ms\":369.3175700000211}}")
prompt = RECORDED_PROMPT['prompt']
print('RECORDED INPUT — no inference in this cell')
for message in prompt['messages']:
    print(message['role'].upper() + ':', message['content'])
print('\nEXACT RENDERED CHAT (repr exposes newlines):')
print(repr(prompt['rendered_chat']))
print('\nGPU input shape:', [1, prompt['input_token_count']])
print('position | token ID | tokenizer piece | decoded single-ID text')
for token in prompt['input_tokens']:
    print(token['index'], token['id'], repr(token['piece']), repr(token['text']), sep=' | ')
assert len(prompt['input_token_ids']) == prompt['input_token_count']
assert [token['id'] for token in prompt['input_tokens']] == prompt['input_token_ids']


### Reveal one forward at a time

Set REVEAL_OUTPUTS to 1, run the cell, predict the next input ID/shape and retained length, then increase the number. No model executes: you are revealing already recorded evidence. Token IDs are selected output values; choosing an ID does not create its K/V until the next forward consumes it. Special stop tokens can count as selected IDs without adding visible text.

The wall and CUDA-event numbers below are **individual instrumented observations**, not medians, a stable target, kernel-time sums or client streaming timings. Model loading, tokenization, decoding/printing the row and HTTP are not included.


In [ ]:
REVEAL_OUTPUTS = 1  # change to len(RECORDED_PROMPT['replay']['steps']) for the complete replay
replay = RECORDED_PROMPT['replay']
assert 1 <= REVEAL_OUTPUTS <= len(replay['steps'])
for step in replay['steps'][:REVEAL_OUTPUTS]:
    token = step['selected_token']
    print('\nOutput choice', step['output_index'], '| phase:', step['phase'])
    print('forward input shape:', step['input_shape'], '| input IDs:', step['input_token_ids'])
    print('selected:', token['id'], repr(token['text']))
    print('KV positions:', step['kv_tokens_before'], '->', step['kv_tokens_after'])
    print('cumulative text:', repr(step['decoded_prefix']))
    print('wall ms:', round(step['wall_ms'], 3), '| CUDA-event interval ms:', round(step['cuda_event_ms'], 3))
    assert step['kv_tokens_after'] == prompt['input_token_count'] + step['output_index'] - 1
if REVEAL_OUTPUTS == len(replay['steps']):
    print('\nFinal visible output:', repr(replay['output_text']))
    print('Selected IDs:', replay['output_token_ids'], '| stop reason:', replay['stop_reason'])
    assert replay['final_kv_tokens'] == prompt['input_token_count'] + len(replay['output_token_ids']) - 1
else:
    print('\nPredict the next row, then increment REVEAL_OUTPUTS. No GPU is used.')


### Transformer recap A — compute every intermediate operation, CPU only

The following complete decoder is deliberately tiny: three rows, four features, one head, a six-feature gated MLP and five output labels A–E. **Invented weights; not a trained model, not Qwen activations, and not an explanation that label D means 8.** No package installation, network, GPU, model files or extra submission artifact is required. Code and fixture are shared with the Book and slides.

Start with STAGE='norm1', inspect all three rows, predict the next stage and rerun with another key from AVAILABLE_STAGES. Every matrix is available in TOY; use TOY['weights'] to inspect all complete weights. Rows are positions, written linear W is [in,out]; PyTorch stores [out,in] and applies its transpose. RoPE pairs (0,2) and (1,3), uses toy theta 10000 (real Qwen theta 1000000), and does not rotate V.

Follow: RMSNorm → Q/K/V+bias → split-half RoPE → scaled QK scores → causal mask → stable softmax → weighted V → output projection → **first residual addition** → second RMSNorm → gate/up → SiLU × up → down → **second residual addition** → final norm → logits → greedy choice. One toy head makes reshape/concat identities; the separate real-shape ledger explains 12 Q heads sharing two KV heads.

Before continuing, explain why normalization does not mix positions, why future probabilities are zero, which tensor bypasses each branch, and why softmax over keys is not softmax over the vocabulary. Full hand calculations and answers are in the Book's transformer recap.


In [ ]:
# Embedded canonical source is isolated from the later required GPU-lab namespace.
recap_namespace = {'__name__': 'topic02_cpu_recap'}
exec("\"\"\"Complete, deliberately tiny decoder arithmetic; standard library only.\n\nThis is an illustrative one-layer/one-head model with invented weights, not Qwen\nactivations, a trained language model, or a hardware/performance simulation.\nRows are positions; linear weights use [input_features, output_features].\nPyTorch Linear stores their transpose. RoPE uses Qwen's split-half pairing.\n\"\"\"\nimport json\nimport math\n\n\ndef linear(rows, weights, bias=None):\n    width = len(weights[0])\n    if not rows or any(len(row) != len(weights) for row in rows):\n        raise ValueError(\"Linear input/weight dimensions disagree\")\n    if any(len(row) != width for row in weights):\n        raise ValueError(\"Ragged weight matrix\")\n    bias = [0.0] * width if bias is None else bias\n    if len(bias) != width:\n        raise ValueError(\"Bias width disagrees\")\n    return [[sum(x * weights[i][j] for i, x in enumerate(row)) + bias[j]\n             for j in range(width)] for row in rows]\n\n\ndef rmsnorm(rows, gain, epsilon=1e-6):\n    if epsilon <= 0 or not math.isfinite(epsilon):\n        raise ValueError(\"epsilon must be positive and finite\")\n    if any(len(row) != len(gain) for row in rows):\n        raise ValueError(\"Gain width disagrees\")\n    return [[x / math.sqrt(sum(y*y for y in row)/len(row) + epsilon) * gain[j]\n             for j, x in enumerate(row)] for row in rows]\n\n\ndef rope(rows, positions, theta=10000.0):\n    \"\"\"Pair first/second halves, as rotate_half does; no learned position table.\"\"\"\n    d = len(rows[0])\n    if d % 2 or len(rows) != len(positions) or any(len(r) != d for r in rows):\n        raise ValueError(\"RoPE needs equal, even rows and one position per row\")\n    out = []\n    for row, position in zip(rows, positions):\n        half = d // 2\n        angles = [position / theta ** (2*i/d) for i in range(half)]\n        out.append([row[i]*math.cos(angles[i])-row[i+half]*math.sin(angles[i])\n                    for i in range(half)] +\n                   [row[i+half]*math.cos(angles[i])+row[i]*math.sin(angles[i])\n                    for i in range(half)])\n    return out\n\n\ndef softmax(row):\n    \"\"\"None is a masked position; JSON never contains Infinity/NaN.\"\"\"\n    active = [x for x in row if x is not None]\n    if not active or not all(math.isfinite(x) for x in active):\n        raise ValueError(\"Softmax needs at least one finite unmasked value\")\n    maximum = max(active)\n    exps = [0.0 if x is None else math.exp(x-maximum) for x in row]\n    total = sum(exps)\n    return [x/total for x in exps]\n\n\ndef add(left, right):\n    if len(left) != len(right) or any(len(a) != len(b) for a,b in zip(left,right)):\n        raise ValueError(\"Residual shapes disagree\")\n    return [[a+b for a,b in zip(x,y)] for x,y in zip(left,right)]\n\n\ndef silu(x):\n    return x/(1+math.exp(-x)) if x >= 0 else x*math.exp(x)/(1+math.exp(x))\n\n\ndef toy_example():\n    x = [[1.,0.,1.,0.], [0.,1.,0.,1.], [1.,2.,3.,4.]]\n    identity = [[float(i == j) for j in range(4)] for i in range(4)]\n    weights = {\n        \"q\": identity, \"q_bias\": [.1,0.,-.1,0.],\n        \"k\": [[identity[i][j]*(.5 if j%2 == 0 else 1.) for j in range(4)] for i in range(4)],\n        \"k_bias\": [0.,.1,0.,-.1], \"v\": identity, \"v_bias\": [.05,0.,0.,-.05],\n        \"o\": [[value*.5 for value in row] for row in identity],\n        \"gate\": [[1.,0.,0.,0.,.5,0.], [0.,1.,0.,0.,.5,0.],\n                 [0.,0.,1.,0.,0.,.5], [0.,0.,0.,1.,0.,.5]],\n        \"up\": [[.5,0.,0.,0.,1.,0.], [0.,.5,0.,0.,-1.,0.],\n               [0.,0.,.5,0.,0.,1.], [0.,0.,0.,.5,0.,-1.]],\n        \"down\": [[.25,0.,0.,0.], [0.,.25,0.,0.], [0.,0.,.25,0.],\n                 [0.,0.,0.,.25], [.1,.1,0.,0.], [0.,0.,.1,.1]],\n        \"lm_head\": [[1.,0.,0.,0.,.25], [0.,1.,0.,0.,.25],\n                    [0.,0.,1.,0.,.25], [0.,0.,0.,1.,.25]],\n        \"norm1_gain\": [1.]*4, \"norm2_gain\": [1.]*4, \"final_gain\": [1.]*4,\n    }\n    s = {\"norm1\": rmsnorm(x, weights[\"norm1_gain\"])}\n    for name in (\"q\", \"k\", \"v\"):\n        s[name] = linear(s[\"norm1\"], weights[name], weights[name+\"_bias\"])\n    s[\"q_rope\"], s[\"k_rope\"] = rope(s[\"q\"], [0,1,2]), rope(s[\"k\"], [0,1,2])\n    s[\"scores\"] = [[sum(a*b for a,b in zip(q,k))/math.sqrt(4)\n                    for k in s[\"k_rope\"]] for q in s[\"q_rope\"]]\n    s[\"masked_scores\"] = [[v if j <= i else None for j,v in enumerate(row)]\n                          for i,row in enumerate(s[\"scores\"])]\n    s[\"shifted_scores\"] = [[None if value is None else value-max(x for x in row if x is not None)\n                            for value in row] for row in s[\"masked_scores\"]]\n    s[\"exp_scores\"] = [[0. if value is None else math.exp(value) for value in row]\n                       for row in s[\"shifted_scores\"]]\n    s[\"softmax_denominators\"] = [[sum(row)] for row in s[\"exp_scores\"]]\n    s[\"probabilities\"] = [softmax(row) for row in s[\"masked_scores\"]]\n    s[\"attention\"] = linear(s[\"probabilities\"], s[\"v\"])\n    # One head: concatenate is identity. A real multihead tensor is transposed first.\n    s[\"attention_output\"] = linear(s[\"attention\"], weights[\"o\"])\n    s[\"residual1\"] = add(x, s[\"attention_output\"])\n    s[\"norm2\"] = rmsnorm(s[\"residual1\"], weights[\"norm2_gain\"])\n    s[\"gate\"] = linear(s[\"norm2\"], weights[\"gate\"])\n    s[\"up\"] = linear(s[\"norm2\"], weights[\"up\"])\n    s[\"silu\"] = [[silu(v) for v in row] for row in s[\"gate\"]]\n    s[\"product\"] = [[a*b for a,b in zip(g,u)] for g,u in zip(s[\"silu\"],s[\"up\"])]\n    s[\"down\"] = linear(s[\"product\"], weights[\"down\"])\n    s[\"residual2\"] = add(s[\"residual1\"],s[\"down\"])\n    s[\"final_norm\"] = rmsnorm(s[\"residual2\"], weights[\"final_gain\"])\n    s[\"logits\"] = linear(s[\"final_norm\"], weights[\"lm_head\"])\n    s[\"output_probabilities\"] = [softmax(row) for row in s[\"logits\"]]\n    index = max(range(5), key=lambda i: s[\"logits\"][-1][i])\n    return {\"schema_version\": \"topic02-transformer-toy-v1\",\n            \"kind\": \"illustrative arithmetic, not trained or measured model activations\",\n            \"config\": {\"sequence\": 3, \"hidden\": 4, \"heads\": 1, \"head_dim\": 4,\n                       \"intermediate\": 6, \"vocabulary\": 5, \"epsilon\": 1e-6,\n                       \"rope_theta\": 10000},\n            \"conventions\": {\"matrix\": \"rows=positions; linear W=[in,out]\",\n                            \"rope\": \"split halves: coordinate pairs (0,2), (1,3)\",\n                            \"mask\": \"null represents forbidden future score, probability zero\",\n                            \"labels\": \"A-E are arbitrary output labels, not Qwen tokens\"},\n            \"input\": x, \"positions\": [0,1,2], \"weights\": weights, \"stages\": s,\n            \"output\": {\"position\": 2, \"index\": index, \"label\": \"ABCDE\"[index]}}\n\n\ndef real_shape_ledger():\n    \"\"\"Pinned Qwen config arithmetic; values are dimensions, never sampled activations.\"\"\"\n    rows = []\n    for phase, query, retained in ((\"prefill\",42,42), (\"decode\",1,43)):\n        rows.append({\"phase\": phase, \"input_ids\": [1,query], \"hidden\": [1,query,1536],\n                     \"q_projected\": [1,query,1536], \"kv_projected\": [1,query,256],\n                     \"q_heads\": [1,12,query,128], \"kv_cache\": [1,2,retained,128],\n                     \"conceptual_scores\": [1,12,query,retained],\n                     \"mlp_branches\": [1,query,8960], \"last_logits\": [1,1,151936],\n                     \"all_layers_logical_kv_bytes\": 2*28*2*retained*128*2})\n    return rows\n\n\nif __name__ == \"__main__\":\n    print(json.dumps(toy_example(), indent=2, allow_nan=False))\n", recap_namespace)
TOY = recap_namespace['toy_example']()
REAL_SHAPES = recap_namespace['real_shape_ledger']()
AVAILABLE_STAGES = list(TOY['stages'])
STAGE = 'norm1'  # try q, q_rope, scores, masked_scores, probabilities, residual1, product, residual2, logits
assert STAGE in AVAILABLE_STAGES
print('ILLUSTRATIVE tiny arithmetic; not captured Qwen values')
print('All stages:', AVAILABLE_STAGES)
print('Input:', TOY['input'])
print('Stage:', STAGE)
for position, row in enumerate(TOY['stages'][STAGE]):
    print(position, [None if value is None else round(value, 6) for value in row])
print('Toy output:', TOY['output'], '(arbitrary A-E labels)')
for row in TOY['stages']['probabilities']:
    assert abs(sum(row) - 1) < 1e-12
assert TOY['stages']['probabilities'][0] == [1.0, 0.0, 0.0]
assert TOY['stages']['residual1'] == recap_namespace['add'](TOY['input'], TOY['stages']['attention_output'])
assert TOY['stages']['residual2'] == recap_namespace['add'](TOY['stages']['residual1'], TOY['stages']['down'])
print('\nSEPARATE real config dimensions, not toy activations:')
for row in REAL_SHAPES:
    print(json.dumps(row))


### Transformer recap B — inspect two recorded real forwards, CPU only

This is a **different evidence type**: actual pinned Qwen2.5-1.5B FP16 values, recorded with the unchanged SDPA callable, from prefill and one decode consuming 8. The last assistant-prefix newline at absolute position 41 predicts 8; the next forward consumes 8 at position 42 and predicts comma. No model executes in this cell. The capture is embedded, so rerunning this viewer never downloads a model or calls an API.

Use STEP=0 or 1 and choose TENSOR from the listed keys. Exact prefix indices describe the selected batch, local position, and head where applicable; feature indices 0–7 identify the displayed slice. All tensors belong to **layer 0 except final_norm**, which is after all 28 blocks. A last-position mean square uses the complete hidden row, not just these eight coordinates.

Top-five probabilities are computed over the full 151936-entry vocabulary and are not renormalized across the five displayed candidates. They are not calibrated answer correctness. The separately stored attention row is **CPU float32 reference reconstruction from captured Q/K/V**, not weights observed inside SDPA. Check its weighted-output error against the capture's declared tolerance; it does not establish kernel allocation or speed. Hooks and CPU copies perturb execution, so this capture makes no latency claim.

Sources and immutable provenance: [recap record](https://levshaazz.github.io/llm-serving-mastery/seminars/runs/2026-10-01-topic02-transformer-recap/README.md). This explanatory route does not add to, shrink, replace or retime the required 50-trial v2 matrix.


In [ ]:
RECORDED_BLOCK = json.loads("{\n  \"protocol\": \"topic02-transformer-recap-v1\",\n  \"scope\": \"one prompt prefill and one decode; educational activation slices, not benchmark\",\n  \"model\": {\n    \"id\": \"Qwen/Qwen2.5-1.5B-Instruct\",\n    \"revision\": \"989aa7980e4cf806f80c7fef2b1adb7bc71aa306\",\n    \"hidden_size\": 1536,\n    \"intermediate_size\": 8960,\n    \"layers\": 28,\n    \"attention_heads\": 12,\n    \"kv_heads\": 2,\n    \"head_dim\": 128,\n    \"vocab_size\": 151936,\n    \"rms_norm_eps\": 1e-06,\n    \"rope_theta\": 1000000.0\n  },\n  \"environment\": {\n    \"torch\": \"2.14.0+cu130\",\n    \"transformers\": \"4.51.3\",\n    \"cuda\": \"13.0\",\n    \"gpu\": \"NVIDIA GeForce RTX 5070 Ti\",\n    \"dtype\": \"float16\",\n    \"attention_requested\": \"sdpa\",\n    \"attention_selected\": \"sdpa\"\n  },\n  \"prompt\": {\n    \"messages\": [\n      {\n        \"role\": \"system\",\n        \"content\": \"You are a helpful assistant. Reply with only the requested numbers.\"\n      },\n      {\n        \"role\": \"user\",\n        \"content\": \"Continue the sequence with three numbers: 2, 4, 6,\"\n      }\n    ],\n    \"rendered_chat\": \"<|im_start|>system\\nYou are a helpful assistant. Reply with only the requested numbers.<|im_end|>\\n<|im_start|>user\\nContinue the sequence with three numbers: 2, 4, 6,<|im_end|>\\n<|im_start|>assistant\\n\",\n    \"input_token_ids\": [\n      151644,\n      8948,\n      198,\n      2610,\n      525,\n      264,\n      10950,\n      17847,\n      13,\n      17841,\n      448,\n      1172,\n      279,\n      11223,\n      5109,\n      13,\n      151645,\n      198,\n      151644,\n      872,\n      198,\n      23526,\n      279,\n      8500,\n      448,\n      2326,\n      5109,\n      25,\n      220,\n      17,\n      11,\n      220,\n      19,\n      11,\n      220,\n      21,\n      11,\n      151645,\n      198,\n      151644,\n      77091,\n      198\n    ],\n    \"input_tokens\": [\n      {\n        \"id\": 151644,\n        \"piece\": \"<|im_start|>\",\n        \"text\": \"<|im_start|>\",\n        \"index\": 0\n      },\n      {\n        \"id\": 8948,\n        \"piece\": \"system\",\n        \"text\": \"system\",\n        \"index\": 1\n      },\n      {\n        \"id\": 198,\n        \"piece\": \"Ċ\",\n        \"text\": \"\\n\",\n        \"index\": 2\n      },\n      {\n        \"id\": 2610,\n        \"piece\": \"You\",\n        \"text\": \"You\",\n        \"index\": 3\n      },\n      {\n        \"id\": 525,\n        \"piece\": \"Ġare\",\n        \"text\": \" are\",\n        \"index\": 4\n      },\n      {\n        \"id\": 264,\n        \"piece\": \"Ġa\",\n        \"text\": \" a\",\n        \"index\": 5\n      },\n      {\n        \"id\": 10950,\n        \"piece\": \"Ġhelpful\",\n        \"text\": \" helpful\",\n        \"index\": 6\n      },\n      {\n        \"id\": 17847,\n        \"piece\": \"Ġassistant\",\n        \"text\": \" assistant\",\n        \"index\": 7\n      },\n      {\n        \"id\": 13,\n        \"piece\": \".\",\n        \"text\": \".\",\n        \"index\": 8\n      },\n      {\n        \"id\": 17841,\n        \"piece\": \"ĠReply\",\n        \"text\": \" Reply\",\n        \"index\": 9\n      },\n      {\n        \"id\": 448,\n        \"piece\": \"Ġwith\",\n        \"text\": \" with\",\n        \"index\": 10\n      },\n      {\n        \"id\": 1172,\n        \"piece\": \"Ġonly\",\n        \"text\": \" only\",\n        \"index\": 11\n      },\n      {\n        \"id\": 279,\n        \"piece\": \"Ġthe\",\n        \"text\": \" the\",\n        \"index\": 12\n      },\n      {\n        \"id\": 11223,\n        \"piece\": \"Ġrequested\",\n        \"text\": \" requested\",\n        \"index\": 13\n      },\n      {\n        \"id\": 5109,\n        \"piece\": \"Ġnumbers\",\n        \"text\": \" numbers\",\n        \"index\": 14\n      },\n      {\n        \"id\": 13,\n        \"piece\": \".\",\n        \"text\": \".\",\n        \"index\": 15\n      },\n      {\n        \"id\": 151645,\n        \"piece\": \"<|im_end|>\",\n        \"text\": \"<|im_end|>\",\n        \"index\": 16\n      },\n      {\n        \"id\": 198,\n        \"piece\": \"Ċ\",\n        \"text\": \"\\n\",\n        \"index\": 17\n      },\n      {\n        \"id\": 151644,\n        \"piece\": \"<|im_start|>\",\n        \"text\": \"<|im_start|>\",\n        \"index\": 18\n      },\n      {\n        \"id\": 872,\n        \"piece\": \"user\",\n        \"text\": \"user\",\n        \"index\": 19\n      },\n      {\n        \"id\": 198,\n        \"piece\": \"Ċ\",\n        \"text\": \"\\n\",\n        \"index\": 20\n      },\n      {\n        \"id\": 23526,\n        \"piece\": \"Continue\",\n        \"text\": \"Continue\",\n        \"index\": 21\n      },\n      {\n        \"id\": 279,\n        \"piece\": \"Ġthe\",\n        \"text\": \" the\",\n        \"index\": 22\n      },\n      {\n        \"id\": 8500,\n        \"piece\": \"Ġsequence\",\n        \"text\": \" sequence\",\n        \"index\": 23\n      },\n      {\n        \"id\": 448,\n        \"piece\": \"Ġwith\",\n        \"text\": \" with\",\n        \"index\": 24\n      },\n      {\n        \"id\": 2326,\n        \"piece\": \"Ġthree\",\n        \"text\": \" three\",\n        \"index\": 25\n      },\n      {\n        \"id\": 5109,\n        \"piece\": \"Ġnumbers\",\n        \"text\": \" numbers\",\n        \"index\": 26\n      },\n      {\n        \"id\": 25,\n        \"piece\": \":\",\n        \"text\": \":\",\n        \"index\": 27\n      },\n      {\n        \"id\": 220,\n        \"piece\": \"Ġ\",\n        \"text\": \" \",\n        \"index\": 28\n      },\n      {\n        \"id\": 17,\n        \"piece\": \"2\",\n        \"text\": \"2\",\n        \"index\": 29\n      },\n      {\n        \"id\": 11,\n        \"piece\": \",\",\n        \"text\": \",\",\n        \"index\": 30\n      },\n      {\n        \"id\": 220,\n        \"piece\": \"Ġ\",\n        \"text\": \" \",\n        \"index\": 31\n      },\n      {\n        \"id\": 19,\n        \"piece\": \"4\",\n        \"text\": \"4\",\n        \"index\": 32\n      },\n      {\n        \"id\": 11,\n        \"piece\": \",\",\n        \"text\": \",\",\n        \"index\": 33\n      },\n      {\n        \"id\": 220,\n        \"piece\": \"Ġ\",\n        \"text\": \" \",\n        \"index\": 34\n      },\n      {\n        \"id\": 21,\n        \"piece\": \"6\",\n        \"text\": \"6\",\n        \"index\": 35\n      },\n      {\n        \"id\": 11,\n        \"piece\": \",\",\n        \"text\": \",\",\n        \"index\": 36\n      },\n      {\n        \"id\": 151645,\n        \"piece\": \"<|im_end|>\",\n        \"text\": \"<|im_end|>\",\n        \"index\": 37\n      },\n      {\n        \"id\": 198,\n        \"piece\": \"Ċ\",\n        \"text\": \"\\n\",\n        \"index\": 38\n      },\n      {\n        \"id\": 151644,\n        \"piece\": \"<|im_start|>\",\n        \"text\": \"<|im_start|>\",\n        \"index\": 39\n      },\n      {\n        \"id\": 77091,\n        \"piece\": \"assistant\",\n        \"text\": \"assistant\",\n        \"index\": 40\n      },\n      {\n        \"id\": 198,\n        \"piece\": \"Ċ\",\n        \"text\": \"\\n\",\n        \"index\": 41\n      }\n    ],\n    \"input_token_count\": 42,\n    \"last_input_token\": {\n      \"id\": 198,\n      \"piece\": \"Ċ\",\n      \"text\": \"\\n\",\n      \"index\": 41\n    }\n  },\n  \"method\": {\n    \"model_forward_calls\": 2,\n    \"warmup_calls\": 0,\n    \"output_attentions\": false,\n    \"captured_layer\": 0,\n    \"slice\": \"batch 0, last input position, first eight features; head 0 where applicable\",\n    \"attention\": \"original SDPA callable unchanged; CPU reconstruction of selected row is not observed SDPA weights\",\n    \"timing\": \"no timing claim: CPU copies and hooks perturb execution\",\n    \"generation\": \"greedy; only first two output tokens selected\"\n  },\n  \"steps\": [\n    {\n      \"phase\": \"prefill\",\n      \"input_ids\": [\n        151644,\n        8948,\n        198,\n        2610,\n        525,\n        264,\n        10950,\n        17847,\n        13,\n        17841,\n        448,\n        1172,\n        279,\n        11223,\n        5109,\n        13,\n        151645,\n        198,\n        151644,\n        872,\n        198,\n        23526,\n        279,\n        8500,\n        448,\n        2326,\n        5109,\n        25,\n        220,\n        17,\n        11,\n        220,\n        19,\n        11,\n        220,\n        21,\n        11,\n        151645,\n        198,\n        151644,\n        77091,\n        198\n      ],\n      \"input_shape\": [\n        1,\n        42\n      ],\n      \"positions\": [\n        0,\n        1,\n        2,\n        3,\n        4,\n        5,\n        6,\n        7,\n        8,\n        9,\n        10,\n        11,\n        12,\n        13,\n        14,\n        15,\n        16,\n        17,\n        18,\n        19,\n        20,\n        21,\n        22,\n        23,\n        24,\n        25,\n        26,\n        27,\n        28,\n        29,\n        30,\n        31,\n        32,\n        33,\n        34,\n        35,\n        36,\n        37,\n        38,\n        39,\n        40,\n        41\n      ],\n      \"selected_token\": {\n        \"id\": 23,\n        \"piece\": \"8\",\n        \"text\": \"8\"\n      },\n      \"top5\": [\n        {\n          \"id\": 23,\n          \"piece\": \"8\",\n          \"text\": \"8\",\n          \"logit\": 30.21875,\n          \"probability\": 0.9910440444946289\n        },\n        {\n          \"id\": 24,\n          \"piece\": \"9\",\n          \"text\": \"9\",\n          \"logit\": 24.703125,\n          \"probability\": 0.003987378440797329\n        },\n        {\n          \"id\": 785,\n          \"piece\": \"The\",\n          \"text\": \"The\",\n          \"logit\": 24.359375,\n          \"probability\": 0.0028274746146053076\n        },\n        {\n          \"id\": 16,\n          \"piece\": \"1\",\n          \"text\": \"1\",\n          \"logit\": 23.46875,\n          \"probability\": 0.0011603932362049818\n        },\n        {\n          \"id\": 1249,\n          \"piece\": \"To\",\n          \"text\": \"To\",\n          \"logit\": 22.109375,\n          \"probability\": 0.0002980136196129024\n        }\n      ],\n      \"logits_shape\": [\n        1,\n        1,\n        151936\n      ],\n      \"selection_position\": 41,\n      \"layer0\": {\n        \"tensors\": {\n          \"attention_concat\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.1485595703125,\n              -0.015289306640625,\n              0.0017118453979492188,\n              -0.0246124267578125,\n              -0.05096435546875,\n              -0.047332763671875,\n              0.1868896484375,\n              -0.036773681640625\n            ]\n          },\n          \"attention_heads\": {\n            \"shape\": [\n              1,\n              12,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.1485595703125,\n              -0.015289306640625,\n              0.0017118453979492188,\n              -0.0246124267578125,\n              -0.05096435546875,\n              -0.047332763671875,\n              0.1868896484375,\n              -0.036773681640625\n            ]\n          },\n          \"attention_output\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.26123046875,\n              -0.010101318359375,\n              -0.01291656494140625,\n              -0.11529541015625,\n              0.43017578125,\n              -0.13134765625,\n              0.024139404296875,\n              0.04931640625\n            ]\n          },\n          \"down\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.0095367431640625,\n              0.01050567626953125,\n              -0.023895263671875,\n              -0.1434326171875,\n              -0.315185546875,\n              0.0139923095703125,\n              -0.11260986328125,\n              -0.1947021484375\n            ]\n          },\n          \"final_norm\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.7451171875,\n              -0.88671875,\n              1.41796875,\n              -0.1888427734375,\n              4.17578125,\n              -1.494140625,\n              0.751953125,\n              -7.11328125\n            ]\n          },\n          \"gate\": {\n            \"shape\": [\n              1,\n              42,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.86328125,\n              0.165771484375,\n              0.1639404296875,\n              -0.0902099609375,\n              -0.268798828125,\n              -1.111328125,\n              -0.1646728515625,\n              -0.333251953125\n            ]\n          },\n          \"input\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.035888671875,\n              -0.048095703125,\n              0.0166015625,\n              0.00836181640625,\n              0.00872802734375,\n              0.031982421875,\n              0.0108642578125,\n              -0.01177978515625\n            ]\n          },\n          \"k_heads\": {\n            \"shape\": [\n              1,\n              2,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              1.306640625,\n              0.65234375,\n              -1.08984375,\n              -0.578125,\n              1.9091796875,\n              0.120361328125,\n              -0.201416015625,\n              -1.8837890625\n            ]\n          },\n          \"k_projection\": {\n            \"shape\": [\n              1,\n              42,\n              256\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              1.306640625,\n              0.65234375,\n              -1.08984375,\n              -0.578125,\n              1.9091796875,\n              0.120361328125,\n              -0.201416015625,\n              -1.8837890625\n            ]\n          },\n          \"k_rotated\": {\n            \"shape\": [\n              1,\n              2,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -1.5390625,\n              1.0400390625,\n              1.162109375,\n              0.53125,\n              0.1741943359375,\n              -0.48876953125,\n              -0.002288818359375,\n              1.98046875\n            ]\n          },\n          \"norm1\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.58349609375,\n              -0.41845703125,\n              0.2314453125,\n              0.1370849609375,\n              0.093017578125,\n              0.55224609375,\n              0.06988525390625,\n              -0.121826171875\n            ]\n          },\n          \"norm2\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.25634765625,\n              -0.12188720703125,\n              0.005016326904296875,\n              -0.2239990234375,\n              0.0007762908935546875,\n              -0.128662109375,\n              0.0738525390625,\n              0.08087158203125\n            ]\n          },\n          \"product\": {\n            \"shape\": [\n              1,\n              42,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.184814453125,\n              -0.00392913818359375,\n              -0.032470703125,\n              0.0026378631591796875,\n              0.0011091232299804688,\n              -0.034454345703125,\n              -0.004764556884765625,\n              -0.029815673828125\n            ]\n          },\n          \"q_heads\": {\n            \"shape\": [\n              1,\n              12,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.019439697265625,\n              0.1796875,\n              0.10882568359375,\n              0.134765625,\n              -0.09454345703125,\n              -0.44287109375,\n              0.1414794921875,\n              0.2337646484375\n            ]\n          },\n          \"q_projection\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.019439697265625,\n              0.1796875,\n              0.10882568359375,\n              0.134765625,\n              -0.09454345703125,\n              -0.44287109375,\n              0.1414794921875,\n              0.2337646484375\n            ]\n          },\n          \"q_rotated\": {\n            \"shape\": [\n              1,\n              12,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.00482177734375,\n              -0.2113037109375,\n              -0.050933837890625,\n              -0.1104736328125,\n              0.1329345703125,\n              -0.036865234375,\n              -0.03399658203125,\n              -0.2427978515625\n            ]\n          },\n          \"residual1\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.225341796875,\n              -0.058197021484375,\n              0.00368499755859375,\n              -0.10693359375,\n              0.43896484375,\n              -0.099365234375,\n              0.035003662109375,\n              0.03753662109375\n            ]\n          },\n          \"residual2\": {\n            \"shape\": [\n              1,\n              42,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.23486328125,\n              -0.047698974609375,\n              -0.02020263671875,\n              -0.25048828125,\n              0.123779296875,\n              -0.08538818359375,\n              -0.07763671875,\n              -0.1572265625\n            ]\n          },\n          \"silu\": {\n            \"shape\": [\n              1,\n              42,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.256103515625,\n              0.0897216796875,\n              0.08868408203125,\n              -0.043060302734375,\n              -0.116455078125,\n              -0.275146484375,\n              -0.0755615234375,\n              -0.13916015625\n            ]\n          },\n          \"up\": {\n            \"shape\": [\n              1,\n              42,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.7216796875,\n              -0.043792724609375,\n              -0.3662109375,\n              -0.061248779296875,\n              -0.009521484375,\n              0.125244140625,\n              0.06304931640625,\n              0.2142333984375\n            ]\n          },\n          \"v_heads\": {\n            \"shape\": [\n              1,\n              2,\n              42,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.1021728515625,\n              -0.15283203125,\n              -0.0257110595703125,\n              0.11419677734375,\n              0.1429443359375,\n              0.0780029296875,\n              0.21630859375,\n              -0.14404296875\n            ]\n          },\n          \"v_projection\": {\n            \"shape\": [\n              1,\n              42,\n              256\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              41\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.1021728515625,\n              -0.15283203125,\n              -0.0257110595703125,\n              0.11419677734375,\n              0.1429443359375,\n              0.0780029296875,\n              0.21630859375,\n              -0.14404296875\n            ]\n          }\n        },\n        \"rmsnorm\": [\n          {\n            \"input\": \"input\",\n            \"output\": \"norm1\",\n            \"epsilon\": 1e-06,\n            \"mean_square\": 0.0009586329106241465,\n            \"inverse_rms\": 32.28103256225586,\n            \"weight_first8\": [\n              0.50390625,\n              0.26953125,\n              0.431640625,\n              0.5078125,\n              0.330078125,\n              0.53515625,\n              0.19921875,\n              0.3203125\n            ]\n          },\n          {\n            \"input\": \"residual1\",\n            \"output\": \"norm2\",\n            \"epsilon\": 1e-06,\n            \"mean_square\": 0.06815367937088013,\n            \"inverse_rms\": 3.8304710388183594,\n            \"weight_first8\": [\n              0.296875,\n              0.546875,\n              0.35546875,\n              0.546875,\n              0.000461578369140625,\n              0.337890625,\n              0.55078125,\n              0.5625\n            ]\n          }\n        ],\n        \"rope_pairs\": [\n          {\n            \"tensor\": \"q\",\n            \"position\": 41,\n            \"head\": 0,\n            \"feature_indices\": [\n              0,\n              64\n            ],\n            \"before\": [\n              0.019439697265625,\n              0.1514892578125\n            ],\n            \"cos\": -0.9873046875,\n            \"sin\": -0.1585693359375,\n            \"after\": [\n              0.00482177734375,\n              -0.152587890625\n            ]\n          },\n          {\n            \"tensor\": \"k\",\n            \"position\": 41,\n            \"head\": 0,\n            \"feature_indices\": [\n              0,\n              64\n            ],\n            \"before\": [\n              1.306640625,\n              -1.5703125\n            ],\n            \"cos\": -0.9873046875,\n            \"sin\": -0.1585693359375,\n            \"after\": [\n              -1.5390625,\n              1.34375\n            ]\n          }\n        ],\n        \"checks\": [\n          {\n            \"operation\": \"residual1 FP16 add\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0\n          },\n          {\n            \"operation\": \"residual2 FP16 add\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0\n          },\n          {\n            \"operation\": \"SiLU\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"gate product\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"first RMSNorm\",\n            \"max_abs_error\": 0.0009765625,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"second RMSNorm\",\n            \"max_abs_error\": 0.000244140625,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"q_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00040346384048461914,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"k_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.0009522438049316406,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"v_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00017079710960388184,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"attention_output first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00012004375457763672,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"gate first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00036084651947021484,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"up first8 CPU FP32 linear\",\n            \"max_abs_error\": 9.942054748535156e-05,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"down first8 CPU FP32 linear\",\n            \"max_abs_error\": 6.575882434844971e-05,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"q RoPE FP16\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.008\n          },\n          {\n            \"operation\": \"k RoPE FP16\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.008\n          },\n          {\n            \"operation\": \"selected attention row CPU FP32 vs SDPA\",\n            \"max_abs_error\": 0.0003616809844970703,\n            \"tolerance\": 0.025\n          }\n        ],\n        \"attention_reconstruction\": {\n          \"kind\": \"CPU FP32 reconstruction; not observed SDPA weights\",\n          \"query_head\": 0,\n          \"kv_head\": 0,\n          \"query_position\": 41,\n          \"key_positions\": [\n            0,\n            1,\n            2,\n            3,\n            4,\n            5,\n            6,\n            7,\n            8,\n            9,\n            10,\n            11,\n            12,\n            13,\n            14,\n            15,\n            16,\n            17,\n            18,\n            19,\n            20,\n            21,\n            22,\n            23,\n            24,\n            25,\n            26,\n            27,\n            28,\n            29,\n            30,\n            31,\n            32,\n            33,\n            34,\n            35,\n            36,\n            37,\n            38,\n            39,\n            40,\n            41\n          ],\n          \"causal_mask\": \"last query can attend to every recorded key; no padding\",\n          \"raw_scores\": [\n            6170.806640625,\n            6157.337890625,\n            6216.53759765625,\n            6233.6708984375,\n            6204.4755859375,\n            6209.26611328125,\n            6196.7490234375,\n            6138.056640625,\n            6211.25390625,\n            6186.5908203125,\n            6197.23779296875,\n            6192.154296875,\n            6210.8671875,\n            6187.0146484375,\n            6173.5849609375,\n            6213.3759765625,\n            6187.62451171875,\n            6219.51611328125,\n            6174.0830078125,\n            6192.23291015625,\n            6220.0947265625,\n            6192.66650390625,\n            6213.01953125,\n            6183.0859375,\n            6201.6279296875,\n            6186.296875,\n            6177.06396484375,\n            6220.3671875,\n            6211.12353515625,\n            6162.34765625,\n            6208.84912109375,\n            6210.9306640625,\n            6156.859375,\n            6208.357421875,\n            6210.7353515625,\n            6163.6083984375,\n            6208.2421875,\n            6191.384765625,\n            6218.60595703125,\n            6171.541015625,\n            6168.61083984375,\n            6217.74462890625\n          ],\n          \"scale\": 0.08838834764831845,\n          \"scaled_scores\": [\n            545.4274291992188,\n            544.2369384765625,\n            549.469482421875,\n            550.98388671875,\n            548.4033813476562,\n            548.8267822265625,\n            547.7203979492188,\n            542.53271484375,\n            549.0025024414062,\n            546.8225708007812,\n            547.7636108398438,\n            547.3142700195312,\n            548.9683227539062,\n            546.8599853515625,\n            545.6729736328125,\n            549.1900634765625,\n            546.9139404296875,\n            549.7327880859375,\n            545.7169799804688,\n            547.3212280273438,\n            549.7838745117188,\n            547.3595581054688,\n            549.1585693359375,\n            546.5127563476562,\n            548.1516723632812,\n            546.7965698242188,\n            545.98046875,\n            549.8079833984375,\n            548.990966796875,\n            544.6797485351562,\n            548.7899169921875,\n            548.9739379882812,\n            544.1946411132812,\n            548.7464599609375,\n            548.9566650390625,\n            544.7911987304688,\n            548.7362670898438,\n            547.2462768554688,\n            549.65234375,\n            545.4923095703125,\n            545.2333374023438,\n            549.576171875\n          ],\n          \"probabilities\": [\n            0.0008604162139818072,\n            0.00026162847643718123,\n            0.04899480193853378,\n            0.2227652370929718,\n            0.016871284693479538,\n            0.02576492168009281,\n            0.008521818555891514,\n            4.7593770432285964e-05,\n            0.030714478343725204,\n            0.003472249023616314,\n            0.008898142725229263,\n            0.005677447654306889,\n            0.029682405292987823,\n            0.0036046227905899286,\n            0.0010998847428709269,\n            0.0370509959757328,\n            0.0038044529501348734,\n            0.06375323235988617,\n            0.001149367424659431,\n            0.005717088934034109,\n            0.06709478050470352,\n            0.005940479692071676,\n            0.03590229153633118,\n            0.002547183306887746,\n            0.013116934336721897,\n            0.003383130766451359,\n            0.0014958588872104883,\n            0.06873201578855515,\n            0.03036220371723175,\n            0.0004073755699209869,\n            0.024832388386130333,\n            0.029849549755454063,\n            0.00025079306215047836,\n            0.023776357993483543,\n            0.02933838590979576,\n            0.00045540439896285534,\n            0.023535238578915596,\n            0.005304251331835985,\n            0.058825504034757614,\n            0.0009180910419672728,\n            0.000708623556420207,\n            0.05451106280088425\n          ],\n          \"weighted_v_first8\": [\n            -0.14861132204532623,\n            -0.01528811827301979,\n            0.0017136811511591077,\n            -0.02460857853293419,\n            -0.05096342787146568,\n            -0.04733005166053772,\n            0.18689261376857758,\n            -0.03678952530026436\n          ]\n        }\n      },\n      \"cache_layers\": [\n        {\n          \"layer\": 0,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 1,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 2,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 3,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 4,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 5,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 6,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 7,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 8,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 9,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 10,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 11,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 12,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 13,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 14,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 15,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 16,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 17,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 18,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 19,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 20,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 21,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 22,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 23,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 24,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 25,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 26,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        },\n        {\n          \"layer\": 27,\n          \"tokens\": 42,\n          \"k_shape\": [\n            1,\n            2,\n            42,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            42,\n            128\n          ]\n        }\n      ]\n    },\n    {\n      \"phase\": \"decode\",\n      \"input_ids\": [\n        23\n      ],\n      \"input_shape\": [\n        1,\n        1\n      ],\n      \"positions\": [\n        42\n      ],\n      \"selected_token\": {\n        \"id\": 11,\n        \"piece\": \",\",\n        \"text\": \",\"\n      },\n      \"top5\": [\n        {\n          \"id\": 11,\n          \"piece\": \",\",\n          \"text\": \",\",\n          \"logit\": 27.40625,\n          \"probability\": 0.999904990196228\n        },\n        {\n          \"id\": 13,\n          \"piece\": \".\",\n          \"text\": \".\",\n          \"logit\": 17.078125,\n          \"probability\": 3.269723310950212e-05\n        },\n        {\n          \"id\": 1154,\n          \"piece\": \"Ġ,\",\n          \"text\": \" ,\",\n          \"logit\": 17.0625,\n          \"probability\": 3.2190313504543155e-05\n        },\n        {\n          \"id\": 151645,\n          \"piece\": \"<|im_end|>\",\n          \"text\": \"<|im_end|>\",\n          \"logit\": 16.5,\n          \"probability\": 1.8341486793360673e-05\n        },\n        {\n          \"id\": 198,\n          \"piece\": \"Ċ\",\n          \"text\": \"\\n\",\n          \"logit\": 15.4296875,\n          \"probability\": 6.28932048130082e-06\n        }\n      ],\n      \"logits_shape\": [\n        1,\n        1,\n        151936\n      ],\n      \"selection_position\": 42,\n      \"layer0\": {\n        \"tensors\": {\n          \"attention_concat\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.053680419921875,\n              0.00832366943359375,\n              0.0321044921875,\n              -0.01444244384765625,\n              0.0255889892578125,\n              0.017333984375,\n              0.193359375,\n              0.04833984375\n            ]\n          },\n          \"attention_heads\": {\n            \"shape\": [\n              1,\n              12,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.053680419921875,\n              0.00832366943359375,\n              0.0321044921875,\n              -0.01444244384765625,\n              0.0255889892578125,\n              0.017333984375,\n              0.193359375,\n              0.04833984375\n            ]\n          },\n          \"attention_output\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.0882568359375,\n              0.1343994140625,\n              -0.07464599609375,\n              -0.128173828125,\n              0.1824951171875,\n              0.18359375,\n              -0.033111572265625,\n              -0.11474609375\n            ]\n          },\n          \"down\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.01611328125,\n              -0.32373046875,\n              -0.02734375,\n              -0.1768798828125,\n              -0.049774169921875,\n              0.429931640625,\n              -0.0247650146484375,\n              -0.10296630859375\n            ]\n          },\n          \"final_norm\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -3.2265625,\n              -0.880859375,\n              -0.006381988525390625,\n              -1.611328125,\n              0.9697265625,\n              0.6591796875,\n              -0.96923828125,\n              -6.375\n            ]\n          },\n          \"gate\": {\n            \"shape\": [\n              1,\n              1,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.7890625,\n              0.60791015625,\n              0.35546875,\n              -0.7509765625,\n              0.07568359375,\n              -1.1962890625,\n              -0.283447265625,\n              -0.070068359375\n            ]\n          },\n          \"input\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.04296875,\n              -0.049072265625,\n              0.017578125,\n              0.045654296875,\n              0.00823974609375,\n              -0.01416015625,\n              0.0306396484375,\n              0.031982421875\n            ]\n          },\n          \"k_heads\": {\n            \"shape\": [\n              1,\n              2,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              9.3828125,\n              1.234375,\n              -0.055145263671875,\n              -0.5810546875,\n              2.8125,\n              -0.362548828125,\n              -0.5380859375,\n              -2.09375\n            ]\n          },\n          \"k_projection\": {\n            \"shape\": [\n              1,\n              1,\n              256\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              9.3828125,\n              1.234375,\n              -0.055145263671875,\n              -0.5810546875,\n              2.8125,\n              -0.362548828125,\n              -0.5380859375,\n              -2.09375\n            ]\n          },\n          \"k_rotated\": {\n            \"shape\": [\n              1,\n              2,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -6.85546875,\n              -1.435546875,\n              2.703125,\n              0.6064453125,\n              1.390625,\n              -0.01422119140625,\n              0.7470703125,\n              2.291015625\n            ]\n          },\n          \"norm1\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.88037109375,\n              -0.53759765625,\n              0.308349609375,\n              0.9423828125,\n              0.11053466796875,\n              -0.30810546875,\n              0.248046875,\n              0.416259765625\n            ]\n          },\n          \"norm2\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.0423583984375,\n              0.1470947265625,\n              -0.06390380859375,\n              -0.1422119140625,\n              0.00027751922607421875,\n              0.1805419921875,\n              -0.004291534423828125,\n              -0.146728515625\n            ]\n          },\n          \"product\": {\n            \"shape\": [\n              1,\n              1,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.1180419921875,\n              -0.11566162109375,\n              -0.07830810546875,\n              0.08966064453125,\n              0.0012464523315429688,\n              -0.010528564453125,\n              0.0206451416015625,\n              0.0007257461547851562\n            ]\n          },\n          \"q_heads\": {\n            \"shape\": [\n              1,\n              12,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.6494140625,\n              0.60498046875,\n              0.3818359375,\n              0.287841796875,\n              0.338134765625,\n              -0.99072265625,\n              -0.1431884765625,\n              -0.051239013671875\n            ]\n          },\n          \"q_projection\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.6494140625,\n              0.60498046875,\n              0.3818359375,\n              0.287841796875,\n              0.338134765625,\n              -0.99072265625,\n              -0.1431884765625,\n              -0.051239013671875\n            ]\n          },\n          \"q_rotated\": {\n            \"shape\": [\n              1,\n              12,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.0924072265625,\n              -1.2197265625,\n              0.0782470703125,\n              -0.269287109375,\n              0.35693359375,\n              0.03759765625,\n              0.7333984375,\n              0.1083984375\n            ]\n          },\n          \"residual1\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.0452880859375,\n              0.0853271484375,\n              -0.05706787109375,\n              -0.08251953125,\n              0.190673828125,\n              0.16943359375,\n              -0.002471923828125,\n              -0.082763671875\n            ]\n          },\n          \"residual2\": {\n            \"shape\": [\n              1,\n              1,\n              1536\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.0291748046875,\n              -0.2384033203125,\n              -0.08441162109375,\n              -0.25927734375,\n              0.140869140625,\n              0.599609375,\n              -0.0272369384765625,\n              -0.185791015625\n            ]\n          },\n          \"silu\": {\n            \"shape\": [\n              1,\n              1,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              -0.2464599609375,\n              0.3935546875,\n              0.208984375,\n              -0.24072265625,\n              0.039276123046875,\n              -0.277587890625,\n              -0.12176513671875,\n              -0.0338134765625\n            ]\n          },\n          \"up\": {\n            \"shape\": [\n              1,\n              1,\n              8960\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.47900390625,\n              -0.2939453125,\n              -0.374755859375,\n              -0.37255859375,\n              0.03173828125,\n              0.037933349609375,\n              -0.1695556640625,\n              -0.0214691162109375\n            ]\n          },\n          \"v_heads\": {\n            \"shape\": [\n              1,\n              2,\n              1,\n              128\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.09710693359375,\n              -0.375244140625,\n              0.6015625,\n              0.1546630859375,\n              0.5478515625,\n              -0.06756591796875,\n              0.318603515625,\n              -0.006885528564453125\n            ]\n          },\n          \"v_projection\": {\n            \"shape\": [\n              1,\n              1,\n              256\n            ],\n            \"slice_prefix_indices\": [\n              0,\n              0\n            ],\n            \"feature_indices\": [\n              0,\n              1,\n              2,\n              3,\n              4,\n              5,\n              6,\n              7\n            ],\n            \"first8\": [\n              0.09710693359375,\n              -0.375244140625,\n              0.6015625,\n              0.1546630859375,\n              0.5478515625,\n              -0.06756591796875,\n              0.318603515625,\n              -0.006885528564453125\n            ]\n          }\n        },\n        \"rmsnorm\": [\n          {\n            \"input\": \"input\",\n            \"output\": \"norm1\",\n            \"epsilon\": 1e-06,\n            \"mean_square\": 0.00060423364629969,\n            \"inverse_rms\": 40.64793395996094,\n            \"weight_first8\": [\n              0.50390625,\n              0.26953125,\n              0.431640625,\n              0.5078125,\n              0.330078125,\n              0.53515625,\n              0.19921875,\n              0.3203125\n            ]\n          },\n          {\n            \"input\": \"residual1\",\n            \"output\": \"norm2\",\n            \"epsilon\": 1e-06,\n            \"mean_square\": 0.1006665825843811,\n            \"inverse_rms\": 3.1517748832702637,\n            \"weight_first8\": [\n              0.296875,\n              0.546875,\n              0.35546875,\n              0.546875,\n              0.000461578369140625,\n              0.337890625,\n              0.55078125,\n              0.5625\n            ]\n          }\n        ],\n        \"rope_pairs\": [\n          {\n            \"tensor\": \"q\",\n            \"position\": 42,\n            \"head\": 0,\n            \"feature_indices\": [\n              0,\n              64\n            ],\n            \"before\": [\n              0.6494140625,\n              0.1826171875\n            ],\n            \"cos\": -0.39990234375,\n            \"sin\": -0.91650390625,\n            \"after\": [\n              -0.0924072265625,\n              -0.66845703125\n            ]\n          },\n          {\n            \"tensor\": \"k\",\n            \"position\": 42,\n            \"head\": 0,\n            \"feature_indices\": [\n              0,\n              64\n            ],\n            \"before\": [\n              9.3828125,\n              -3.38671875\n            ],\n            \"cos\": -0.39990234375,\n            \"sin\": -0.91650390625,\n            \"after\": [\n              -6.85546875,\n              -7.24609375\n            ]\n          }\n        ],\n        \"checks\": [\n          {\n            \"operation\": \"residual1 FP16 add\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0\n          },\n          {\n            \"operation\": \"residual2 FP16 add\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0\n          },\n          {\n            \"operation\": \"SiLU\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"gate product\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"first RMSNorm\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"second RMSNorm\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.004\n          },\n          {\n            \"operation\": \"q_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00022876262664794922,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"k_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.0026369094848632812,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"v_projection first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.00018072128295898438,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"attention_output first8 CPU FP32 linear\",\n            \"max_abs_error\": 3.123283386230469e-05,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"gate first8 CPU FP32 linear\",\n            \"max_abs_error\": 0.0004082918167114258,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"up first8 CPU FP32 linear\",\n            \"max_abs_error\": 7.987022399902344e-05,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"down first8 CPU FP32 linear\",\n            \"max_abs_error\": 4.9173831939697266e-05,\n            \"tolerance\": 0.05\n          },\n          {\n            \"operation\": \"q RoPE FP16\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.008\n          },\n          {\n            \"operation\": \"k RoPE FP16\",\n            \"max_abs_error\": 0.0,\n            \"tolerance\": 0.008\n          },\n          {\n            \"operation\": \"selected attention row CPU FP32 vs SDPA\",\n            \"max_abs_error\": 0.0008528232574462891,\n            \"tolerance\": 0.025\n          }\n        ],\n        \"attention_reconstruction\": {\n          \"kind\": \"CPU FP32 reconstruction; not observed SDPA weights\",\n          \"query_head\": 0,\n          \"kv_head\": 0,\n          \"query_position\": 42,\n          \"key_positions\": [\n            0,\n            1,\n            2,\n            3,\n            4,\n            5,\n            6,\n            7,\n            8,\n            9,\n            10,\n            11,\n            12,\n            13,\n            14,\n            15,\n            16,\n            17,\n            18,\n            19,\n            20,\n            21,\n            22,\n            23,\n            24,\n            25,\n            26,\n            27,\n            28,\n            29,\n            30,\n            31,\n            32,\n            33,\n            34,\n            35,\n            36,\n            37,\n            38,\n            39,\n            40,\n            41,\n            42\n          ],\n          \"causal_mask\": \"last query can attend to every recorded key; no padding\",\n          \"raw_scores\": [\n            5842.2021484375,\n            5858.2578125,\n            5889.4833984375,\n            5894.427734375,\n            5897.23486328125,\n            5900.5673828125,\n            5861.53759765625,\n            5831.4208984375,\n            5900.330078125,\n            5873.5185546875,\n            5883.8720703125,\n            5868.6220703125,\n            5902.8046875,\n            5866.04638671875,\n            5864.49609375,\n            5905.26953125,\n            5866.01953125,\n            5896.71923828125,\n            5850.501953125,\n            5868.87109375,\n            5897.197265625,\n            5861.8759765625,\n            5905.77783203125,\n            5870.546875,\n            5892.5576171875,\n            5873.5458984375,\n            5867.892578125,\n            5907.23095703125,\n            5891.8857421875,\n            5880.4873046875,\n            5901.94091796875,\n            5891.4736328125,\n            5872.9892578125,\n            5899.6337890625,\n            5893.33984375,\n            5875.7919921875,\n            5902.71240234375,\n            5867.7998046875,\n            5897.7373046875,\n            5844.9306640625,\n            5865.4443359375,\n            5896.11767578125,\n            5899.08642578125\n          ],\n          \"scale\": 0.08838834764831845,\n          \"scaled_scores\": [\n            516.3826293945312,\n            517.8017578125,\n            520.5617065429688,\n            520.9987182617188,\n            521.246826171875,\n            521.5413818359375,\n            518.0916137695312,\n            515.4296875,\n            521.5204467773438,\n            519.150634765625,\n            520.0657348632812,\n            518.7178344726562,\n            521.7391357421875,\n            518.4901733398438,\n            518.3531494140625,\n            521.95703125,\n            518.48779296875,\n            521.2012939453125,\n            517.1162109375,\n            518.7398071289062,\n            521.2435302734375,\n            518.1215209960938,\n            522.001953125,\n            518.887939453125,\n            520.8334350585938,\n            519.1530151367188,\n            518.6533203125,\n            522.13037109375,\n            520.7740478515625,\n            519.7665405273438,\n            521.662841796875,\n            520.7376098632812,\n            519.1038208007812,\n            521.4588623046875,\n            520.902587890625,\n            519.3515625,\n            521.7310180664062,\n            518.6451416015625,\n            521.291259765625,\n            516.623779296875,\n            518.4369506835938,\n            521.1481323242188,\n            521.4105224609375\n          ],\n          \"probabilities\": [\n            0.0002852118923328817,\n            0.0011789279524236917,\n            0.01862592250108719,\n            0.028834309428930283,\n            0.03695400059223175,\n            0.04961184039711952,\n            0.0015753248007968068,\n            0.00010997911158483475,\n            0.04858401045203209,\n            0.004542522598057985,\n            0.011342792771756649,\n            0.002946689026430249,\n            0.06046009063720703,\n            0.002346725668758154,\n            0.0020462260581552982,\n            0.07517952471971512,\n            0.0023411461152136326,\n            0.03530913218855858,\n            0.0005939611000940204,\n            0.003012151923030615,\n            0.036832403391599655,\n            0.0016231500776484609,\n            0.07863374054431915,\n            0.0034930913243442774,\n            0.02444150485098362,\n            0.004553348291665316,\n            0.002762588206678629,\n            0.0894087702035904,\n            0.023032253608107567,\n            0.008409719914197922,\n            0.05601892247796059,\n            0.022208109498023987,\n            0.004334769677370787,\n            0.045682258903980255,\n            0.02619151584804058,\n            0.005553399212658405,\n            0.05997127667069435,\n            0.002740086056292057,\n            0.03863302245736122,\n            0.000362992548616603,\n            0.0022250921465456486,\n            0.033481061458587646,\n            0.043526507914066315\n          ],\n          \"weighted_v_first8\": [\n            -0.05367964133620262,\n            0.008329043164849281,\n            0.032095715403556824,\n            -0.01444370485842228,\n            0.025591189041733742,\n            0.017337588593363762,\n            0.19341161847114563,\n            0.04833567515015602\n          ]\n        }\n      },\n      \"cache_layers\": [\n        {\n          \"layer\": 0,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 1,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 2,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 3,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 4,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 5,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 6,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 7,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 8,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 9,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 10,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 11,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 12,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 13,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 14,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 15,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 16,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 17,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 18,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 19,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 20,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 21,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 22,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 23,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 24,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 25,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 26,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        },\n        {\n          \"layer\": 27,\n          \"tokens\": 43,\n          \"k_shape\": [\n            1,\n            2,\n            43,\n            128\n          ],\n          \"v_shape\": [\n            1,\n            2,\n            43,\n            128\n          ]\n        }\n      ]\n    }\n  ],\n  \"resource_samples\": [\n    {\n      \"stage\": \"before_load\",\n      \"host_available_bytes\": 74069581824,\n      \"device_used_bytes\": 1356398592,\n      \"allocated_bytes\": 0,\n      \"reserved_bytes\": 0,\n      \"peak_allocated_bytes\": 0,\n      \"peak_reserved_bytes\": 0\n    },\n    {\n      \"stage\": \"post_load\",\n      \"host_available_bytes\": 71288455168,\n      \"device_used_bytes\": 4642635776,\n      \"allocated_bytes\": 3088346624,\n      \"reserved_bytes\": 3286237184,\n      \"peak_allocated_bytes\": 3088346624,\n      \"peak_reserved_bytes\": 3286237184\n    },\n    {\n      \"stage\": \"after_prefill\",\n      \"host_available_bytes\": 71280308224,\n      \"device_used_bytes\": 4737007616,\n      \"allocated_bytes\": 3123715072,\n      \"reserved_bytes\": 3323985920,\n      \"peak_allocated_bytes\": 3125901824,\n      \"peak_reserved_bytes\": 3323985920\n    },\n    {\n      \"stage\": \"after_decode\",\n      \"host_available_bytes\": 71280685056,\n      \"device_used_bytes\": 4737007616,\n      \"allocated_bytes\": 3123744256,\n      \"reserved_bytes\": 3323985920,\n      \"peak_allocated_bytes\": 3125901824,\n      \"peak_reserved_bytes\": 3323985920\n    },\n    {\n      \"stage\": \"after_release\",\n      \"host_available_bytes\": 71280513024,\n      \"device_used_bytes\": 1446576128,\n      \"allocated_bytes\": 33554432,\n      \"reserved_bytes\": 33554432,\n      \"peak_allocated_bytes\": 3125901824,\n      \"peak_reserved_bytes\": 3323985920\n    }\n  ]\n}\n")
STEP = 0  # 0=prefill; 1=one decode forward
TENSOR = 'norm1'  # try q_projection, q_rotated, attention_output, residual1, gate, silu, product, down, residual2
assert STEP in (0, 1)
block_step = RECORDED_BLOCK['steps'][STEP]
tensor = block_step['layer0']['tensors'][TENSOR]
print('RECORDED REAL CAPTURE — no inference')
print('Literal user prompt:', RECORDED_BLOCK['prompt']['messages'][-1]['content'])
print('Last original input:', RECORDED_BLOCK['prompt']['last_input_token'])
print('Phase/input shape/absolute prediction position:', block_step['phase'], block_step['input_shape'], block_step['selection_position'])
print('Available tensors:', list(block_step['layer0']['tensors']))
print('Scope:', 'after all 28 blocks' if TENSOR == 'final_norm' else 'layer 0')
print(TENSOR, 'shape:', tensor['shape'], 'prefix indices:', tensor['slice_prefix_indices'])
print('Feature indices:', tensor['feature_indices'], 'recorded values:', tensor['first8'])
print('Chosen:', block_step['selected_token'])
print('Full-vocabulary top five (not renormalized):')
for candidate in block_step['top5']:
    print(candidate)
reference_attention = block_step['layer0']['attention_reconstruction']
print('\nSEPARATE:', reference_attention['kind'])
print('Query/KV heads:', reference_attention['query_head'], reference_attention['kv_head'])
for position, score, probability in zip(reference_attention['key_positions'], reference_attention['scaled_scores'], reference_attention['probabilities']):
    print('key', position, 'scaled score', round(score, 6), 'reference probability', round(probability, 8))
assert abs(sum(reference_attention['probabilities']) - 1) < 1e-5
for check in block_step['layer0']['checks']:
    assert check['max_abs_error'] <= check['tolerance'], check
    print('Numerical consistency:', check)
assert len(block_step['cache_layers']) == 28
assert all(layer['tokens'] == 42 + STEP for layer in block_step['cache_layers'])
print('This recorded capture has no timing claim and is not your required GPU evidence.')

# Separate, executed CPU experiments using invented features, not recorded GPU timing.
mechanism_namespace = {'__name__': 'topic02_cpu_mechanisms'}
exec("\"\"\"CPU-only mechanisms: exact online normalization and an executed cache A/B.\n\nInvented one-block inputs/weights. Operation counts are observations of this\nPython program, not GPU timings, hardware traffic, or a trained-model speedup.\n\"\"\"\nimport math\n\n\ndef online_attention(scores, values, blocks):\n    \"\"\"One unmasked query/head; retain max m, denominator l and numerator a.\"\"\"\n    if not scores or len(scores) != len(values) or sum(blocks) != len(scores):\n        raise ValueError('Every score/value must belong to exactly one block')\n    if any(not isinstance(n, int) or n <= 0 for n in blocks):\n        raise ValueError('Block lengths must be positive integers')\n    width = len(values[0])\n    if not width or any(len(row) != width for row in values):\n        raise ValueError('Value widths must agree')\n    if not all(math.isfinite(x) for x in scores + [v for row in values for v in row]):\n        raise ValueError('This example requires finite unmasked values')\n    m, denominator, numerator, offset = -math.inf, 0., [0.]*width, 0\n    states = []\n    for size in blocks:\n        chunk = scores[offset:offset+size]\n        new_m = max(m, max(chunk))\n        rescale = math.exp(m-new_m)  # exp(-inf) = 0 for the first block\n        exps = [math.exp(score-new_m) for score in chunk]\n        denominator = rescale*denominator + sum(exps)\n        numerator = [rescale*old + sum(e*row[r] for e,row in\n                     zip(exps, values[offset:offset+size])) for r,old in enumerate(numerator)]\n        m = new_m\n        states.append({'m':m, 'rescale':rescale, 'denominator':denominator,\n                       'numerator':numerator[:], 'output':[x/denominator for x in numerator]})\n        offset += size\n    return states\n\n\ndef online_example():\n    # Scores 0, ln2, ln4 give exact relative masses 1,2,4.\n    scores = [0., math.log(2), math.log(4)]\n    values = [[1.], [3.], [5.]]\n    return {'kind':'illustrative exact arithmetic, not a GPU observation',\n            'scores':scores, 'values':values, 'blocks':[2,1],\n            'states':online_attention(scores, values, [2,1]), 'full_output':27/7}\n\n\ndef cache_ab(toy, prompt=42, decode=9):\n    \"\"\"Same fixed input rows, weights, positions and outputs; only KV reuse changes.\n\n    Caller supplies the complete public toy fixture. Actual linear operations\n    increment counters, so the result does not simply print a closed-form formula.\n    \"\"\"\n    if prompt < 1 or decode < 1:\n        raise ValueError('Need a nonempty prefix and at least one decode call')\n    weights = toy['weights']\n    rows = [toy['input'][i % len(toy['input'])][:] for i in range(prompt+decode)]\n\n    def run(cached):\n        count = 0\n        retained_k, retained_v, outputs = [], [], []\n        def linear(row, name, bias=None):\n            nonlocal count\n            if name in ('q','k','v'):\n                count += 1\n            w = weights[name]\n            return [sum(x*w[i][j] for i,x in enumerate(row)) + (bias[j] if bias else 0.)\n                    for j in range(len(w[0]))]\n        def norm(row, gain):\n            inv = 1/math.sqrt(sum(x*x for x in row)/len(row)+1e-6)\n            return [x*inv*g for x,g in zip(row,weights[gain])]\n        def rotate(row, position):\n            half = len(row)//2\n            angles = [position/10000**(2*i/len(row)) for i in range(half)]\n            return [row[i]*math.cos(a)-row[i+half]*math.sin(a) for i,a in enumerate(angles)] + [\n                    row[i+half]*math.cos(a)+row[i]*math.sin(a) for i,a in enumerate(angles)]\n        for length in range(prompt, prompt+decode+1):\n            begin = len(retained_k) if cached else 0\n            if not cached:\n                retained_k, retained_v = [], []\n            q = None\n            for position in range(begin, length):\n                h1 = norm(rows[position], 'norm1_gain')\n                q = rotate(linear(h1,'q',weights['q_bias']),position)\n                retained_k.append(rotate(linear(h1,'k',weights['k_bias']),position))\n                retained_v.append(linear(h1,'v',weights['v_bias']))\n            # The last position can attend every retained key, including itself.\n            scores = [sum(a*b for a,b in zip(q,k))/2 for k in retained_k]\n            maximum = max(scores)\n            exps = [math.exp(s-maximum) for s in scores]\n            probabilities = [e/sum(exps) for e in exps]\n            attention = [sum(p*v[r] for p,v in zip(probabilities,retained_v)) for r in range(4)]\n            branch = linear(attention,'o')\n            x1 = [a+b for a,b in zip(rows[length-1],branch)]\n            h2 = norm(x1,'norm2_gain')\n            gate, up = linear(h2,'gate'), linear(h2,'up')\n            product = [g/(1+math.exp(-g))*u for g,u in zip(gate,up)]\n            down = linear(product,'down')\n            x2 = [a+b for a,b in zip(x1,down)]\n            outputs.append(linear(norm(x2,'final_gain'),'lm_head'))\n        return {'qkv_projection_row_calls':count, 'retained_positions':len(retained_k), 'logits':outputs}\n    baseline, cached = run(False), run(True)\n    error = max(abs(a-b) for x,y in zip(baseline['logits'],cached['logits']) for a,b in zip(x,y))\n    return {'kind':'executed CPU operation-count experiment; no latency claim',\n            'prompt':prompt, 'decode_calls':decode, 'baseline':baseline, 'cached':cached,\n            'max_abs_logit_difference':error}\n", mechanism_namespace)
ONLINE = mechanism_namespace['online_example']()
print('ONLINE SOFTMAX: exact two-block arithmetic', ONLINE)
assert abs(ONLINE['states'][-1]['output'][0] - 27/7) < 1e-12
CACHE_AB = mechanism_namespace['cache_ab'](TOY)
print('CPU CACHE A/B: Q/K/V row-projection calls',
      CACHE_AB['baseline']['qkv_projection_row_calls'],
      CACHE_AB['cached']['qkv_projection_row_calls'])
print('Maximum logit difference:', CACHE_AB['max_abs_logit_difference'])
assert CACHE_AB['baseline']['qkv_projection_row_calls'] == 1395
assert CACHE_AB['cached']['qkv_projection_row_calls'] == 153
assert CACHE_AB['max_abs_logit_difference'] < 1e-12
print('Removed 1242 redundant row projections; no wall-time or GPU speedup claim.')


### Before the GPU route: mechanisms and three measured comparisons

The preceding cell also executes two separately labelled CPU examples. Online softmax rescales numerator and denominator when the maximum changes: 3.5/1.5 becomes 6.75/1.75 = 27/7. The cache intervention uses the same fixed feature rows in both paths, counts 1395 versus 153 Q/K/V row projections, and checks all ten logit vectors. This is a completed operation-count diagnosis, not GPU timing or completion of the required lab. Read the [worked case and GPU follow-up protocol](https://levshaazz.github.io/llm-serving-mastery/en/book/02/#cache-diagnosis).

Keep tensor names consistent: X=input, h1=norm1, X1=residual1, h2=norm2, X2=residual2. S_q counts new positions; S_kv includes the new decode key. H_q and H_kv are different axes in GQA.

These are canonical **protocol-v2** measurements, not timings for the sentence above. Source: [30 September raw trials and protocol](https://levshaazz.github.io/llm-serving-mastery/seminars/runs/2026-09-30-topic02-protocol-v2/README.md). RTX 5070 Ti, FP16, five trials per case; brackets are observed min–max, not p95 or confidence intervals.

| Pinned 1.5B case | Measured observation | Predict / explain |
|---|---|---|
| B1/P4096/O32 | Prefill 163.7 ms [163.4–163.8]; 31-step decode 1039.1 ms [763.9–1154.4] | Why can 4096 known positions finish before 31 dependent steps? |
| B1→B4, P128/O32 | Aggregate 36.9 [24.7–37.1] → 155.2 [103.3–162.5] tok/s; per-lane cadence 27.1 [26.9–40.5] → 25.8 [24.6–38.7] ms | Reconstruct numerators 31 and 124. Did one lane become 4.2× faster? |
| B1/P4096, O32→O128 | Calculated final KV 112.85 → 115.47 MiB; measured allocation peak 3349.34 MiB in both | Why can final KV grow without a higher whole-phase peak? |

The 1.5B coefficient is 28 KiB/token (28 layers, 2 KV heads, head dimension 128, FP16), **not** the 3B transfer exercise's 36 KiB/token. Final retained length is P+O−1. No row establishes serving capacity or an HTTP SLO.

**Unresolved control:** identical P128/B1 prefills under O32 and O128 labels have medians 22.3 and 64.6 ms, although the future output count is not part of the prefill forward. Preserve this discrepancy rather than inventing its cause.

Only continue below when ready for your own bounded GPU experiment. Recorded replay may be your classroom fallback, never a replacement for required GPU evidence.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess, sys
pinned = {'transformers': '4.51.3', 'accelerate': '1.6.0', 'huggingface_hub': '0.30.2', 'safetensors': '0.5.3'}
for name, wanted in pinned.items():
    try: installed = version(name) == wanted
    except PackageNotFoundError: installed = False
    if not installed:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'{name}=={wanted}'])
# If you previously imported a different Transformers version, restart the kernel now.


## 1. Read the measurement boundary

The following cell is the complete editable implementation. Prefill includes one forward and selection of output y1; decode includes O−1 forwards and greedy selections. The last selected token has no KV until it is fed back, so retained length is P+O−1. EOS is ignored to hold lengths fixed. Inputs are repeated synthetic token IDs, with no padding or chat-template claim.

Each phase records synchronized wall time and a CUDA-event interval. **An event interval is not a sum of kernel durations:** host launch gaps can lie between events. Neither timer includes tokenization, model loading, HTTP, queueing or delivery. Do not call these values server TTFT/TPOT or p95.

Last-position logits avoid computing unused vocabulary projections for every prompt position. Full-shape warmup is outside timing; cases are shuffled within five rounds, one model at a time. Allocator pools stay warm across trials. Post-trial live-allocation recovery and host RAM are checked; peaks are recorded. Gates are stop rules, not an OS memory sandbox. Instructor runs also use an external memory limit and wall timeout.


In [ ]:
"""Topic 02 protocol v2. Embedded verbatim in the standalone public notebook.

No downloads on import; no CUDA required for the arithmetic/contract functions.
This is a guarded Python phase experiment, not an HTTP service benchmark.
"""
import gc
import json
import math
import os
import platform
import random
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

PROTOCOL = "topic02-phase-v2"
MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]
TRIALS = 5
ORDER_SEED = 20260930
GIB = 1024 ** 3


def kv_bytes_per_token(config, element_bytes=2):
    head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
    return 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * element_bytes


def attention_pairs(prompt, output):
    """Decode only: output 1 is selected from prefill; each later query includes itself."""
    steps = output - 1
    return steps * prompt + steps * (steps + 1) // 2


def trial_order(cases=CASES, trials=TRIALS):
    rng = random.Random(ORDER_SEED)
    schedule = []
    for repeat in range(trials):
        order = list(cases)
        rng.shuffle(order)
        schedule.extend((repeat, *case) for case in order)
    return schedule


def checkpoint(path, data):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def host_available_bytes():
    # Colab and the instructor's WSL environment are Linux. Fail closed on unknown hosts.
    meminfo = Path("/proc/meminfo")
    if not meminfo.exists():
        raise RuntimeError("Run this CUDA lab on Linux/Colab/WSL; host memory gate is unavailable")
    for line in meminfo.read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("MemAvailable is unavailable")


def process_rss_bytes():
    return int(Path("/proc/self/statm").read_text().split()[1]) * os.sysconf("SC_PAGE_SIZE")


def resource_gate(torch, baseline_allocated=None, baseline_rss=None):
    allocated = torch.cuda.memory_allocated()
    free, total = torch.cuda.mem_get_info()
    limit = min(12 * GIB, int(total * 0.80))
    if allocated > limit or total - free > min(12 * GIB, int(total * 0.90)):
        raise RuntimeError("STOP: GPU memory budget exceeded; retain partial evidence and inspect other processes")
    if host_available_bytes() < 2 * GIB:
        raise RuntimeError("STOP: less than 2 GiB host RAM available")
    if baseline_allocated is not None and allocated > baseline_allocated + 128 * 1024 ** 2:
        raise RuntimeError("STOP: live CUDA allocations failed to return to the post-load baseline")
    if baseline_rss is not None and process_rss_bytes() > baseline_rss + 2 * GIB:
        raise RuntimeError("STOP: process RSS grew by more than 2 GiB")
    return {"allocated_bytes": allocated, "reserved_bytes": torch.cuda.memory_reserved(),
            "device_used_bytes": total - free, "host_available_bytes": host_available_bytes(),
            "process_rss_bytes": process_rss_bytes()}


def exact_input(torch, tokenizer, batch, length):
    tokens = tokenizer("Explain GPU memory reuse and token dependencies. ", add_special_tokens=False)["input_ids"]
    row = (tokens * math.ceil(length / len(tokens)))[:length]
    return torch.tensor([row] * batch, dtype=torch.long, device="cuda")


def phase_trial(torch, model, ids, output_tokens):
    """No HTTP, tokenization, model load or profiler inside these intervals.

    Prefill includes selecting y1. Decode includes O-1 forwards and greedy selections.
    CUDA-event elapsed time spans host launch gaps too; it is not summed kernel time.
    Exact-length synthetic generation deliberately ignores EOS; it is not a quality test.
    """
    out = past = token = None
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    try:
        with torch.inference_mode():
            wall_start = time.perf_counter()
            start.record()
            out = model(ids, use_cache=True, logits_to_keep=1)
            token = out.logits[:, -1].argmax(-1, keepdim=True)
            past = out.past_key_values
            out = None  # Do not retain a full prompt-logit tensor during decode.
            end.record()
            end.synchronize()
            prefill_wall_ms = (time.perf_counter() - wall_start) * 1000
            prefill_event_ms = start.elapsed_time(end)
            prefill_peak = torch.cuda.max_memory_allocated()
            wall_start = time.perf_counter()
            start.record()
            for _ in range(output_tokens - 1):
                out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                token = out.logits[:, -1].argmax(-1, keepdim=True)
                past = out.past_key_values
                out = None
            end.record()
            end.synchronize()
            decode_wall_ms = (time.perf_counter() - wall_start) * 1000
            decode_event_ms = start.elapsed_time(end)
            retained = int(past.get_seq_length())
            expected = ids.shape[1] + output_tokens - 1
            if retained != expected:
                raise AssertionError(f"KV sequence length {retained} != {expected}")
            steps = output_tokens - 1
            return {
                "prefill_wall_ms": prefill_wall_ms, "prefill_event_ms": prefill_event_ms,
                "decode_wall_ms": decode_wall_ms, "decode_event_ms": decode_event_ms,
                "decode_steps_per_sequence": steps, "generated_tokens": ids.shape[0] * output_tokens,
                "retained_tokens_per_sequence": retained,
                "decode_aggregate_tokens_per_s": ids.shape[0] * steps * 1000 / decode_wall_ms,
                "decode_mean_step_ms": decode_wall_ms / steps,
                "prefill_peak_allocated_bytes": prefill_peak,
                "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
                "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
            }
    finally:
        out = past = token = None


def run_matrix(model_paths, output_dir, *, smoke=False):
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required; CPU checks cannot replace this run")
    if transformers.__version__ != "4.51.3":
        raise RuntimeError("Use the pinned Transformers 4.51.3 runtime")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)  # Never silently overwrite a previous attempt.
    output_path = output_dir / "topic02-benchmark.json"
    cases = [CASES[0]] if smoke else CASES
    model_ids = list(MODELS)[:1] if smoke else list(MODELS)
    data = {"protocol": PROTOCOL, "status": "running", "scope": "smoke" if smoke else "core",
            "started_at": datetime.now(timezone.utc).isoformat(),
            "environment": {"python": platform.python_version(), "torch": torch.__version__,
                            "transformers": transformers.__version__, "cuda": torch.version.cuda,
                            "gpu": torch.cuda.get_device_name(), "dtype": "float16",
                            "attention_requested": "sdpa", "logits_to_keep": 1},
            "models": MODELS, "order_seed": ORDER_SEED, "trials": 1 if smoke else TRIALS,
            "cases": [list(c) for c in cases], "results": [], "resource_samples": [],
            "measurement": "synchronized wall + CUDA-event intervals, greedy forced length, no HTTP",
            "model_details": {}}
    checkpoint(output_path, data)
    model = tokenizer = ids = None
    try:
        data["resource_samples"].append(resource_gate(torch))
        for model_id in model_ids:
            try:
                model = AutoModelForCausalLM.from_pretrained(
                    model_paths[model_id], local_files_only=True, torch_dtype=torch.float16,
                    attn_implementation="sdpa").to("cuda").eval()
                tokenizer = AutoTokenizer.from_pretrained(model_paths[model_id], local_files_only=True)
                torch.cuda.synchronize()
                baseline = torch.cuda.memory_allocated()
                rss = process_rss_bytes()
                data["model_details"][model_id] = {
                    "kv_bytes_per_token": kv_bytes_per_token(model.config),
                    "post_load_allocated_bytes": baseline,
                    "attention_selected": model.config._attn_implementation,
                    "vocab_size": model.config.vocab_size,
                }
                resource_gate(torch, baseline, rss)
                # Warm each actual shape, including the complete decode path. Warmup is not a trial.
                for batch, prompt, output in cases:
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    resource_gate(torch, baseline, rss)
                for repeat, batch, prompt, output in trial_order(cases, data["trials"]):
                    resource_gate(torch, baseline, rss)
                    ids = exact_input(torch, tokenizer, batch, prompt)
                    result = phase_trial(torch, model, ids, output)
                    ids = None
                    gc.collect()
                    result.update(model=model_id, batch=batch, input_tokens=prompt,
                                  output_tokens=output, repeat=repeat)
                    data["results"].append(result)
                    checkpoint(output_path, data)  # Preserve the trial before checking recovery.
                    data["resource_samples"].append(resource_gate(torch, baseline, rss))
                    if result["peak_allocated_bytes"] > min(12 * GIB, torch.cuda.get_device_properties(0).total_memory * .80):
                        raise RuntimeError("STOP: an in-process allocation peak exceeded the budget")
                    checkpoint(output_path, data)
                    print(model_id, (batch, prompt, output), "trial", repeat + 1, flush=True)
            finally:
                model = tokenizer = ids = None
                gc.collect()
                torch.cuda.empty_cache()  # Between models only, never between timed trials.
                torch.cuda.synchronize()
                data["resource_samples"].append(resource_gate(torch))
        data["status"] = "complete"
    except Exception as exc:
        data["status"] = "failed"
        data["error_type"] = type(exc).__name__
        raise
    finally:
        data["finished_at"] = datetime.now(timezone.utc).isoformat()
        checkpoint(output_path, data)
    return data


def summarize(data):
    rows = []
    groups = {}
    for trial in data["results"]:
        key = tuple(trial[k] for k in ("model", "batch", "input_tokens", "output_tokens"))
        groups.setdefault(key, []).append(trial)
    for key, trials in groups.items():
        row = dict(zip(("model", "batch", "input_tokens", "output_tokens"), key))
        row["n"] = len(trials)
        for metric in ("prefill_wall_ms", "decode_wall_ms", "decode_mean_step_ms", "decode_aggregate_tokens_per_s"):
            values = [t[metric] for t in trials]
            row[metric] = {"median": statistics.median(values), "min": min(values), "max": max(values)}
        rows.append(row)
    return rows


def focused_profiles(model_path, output_dir):
    """Two independent traces: P=128 and P=4096, each prefill + seven decode steps.

    No profile_memory/record_shapes by default: these add overhead/retain references.
    The trace proves event ordering/operator attribution, not unprofiled latency or HBM bandwidth.
    """
    import torch
    from torch.profiler import profile, record_function, ProfilerActivity
    from transformers import AutoModelForCausalLM, AutoTokenizer
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    model = tokenizer = ids = out = past = token = None
    summaries = []
    try:
        resource_gate(torch)
        model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True,
                    torch_dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        baseline = torch.cuda.memory_allocated()
        for prompt in (128, 4096):
            ids = exact_input(torch, tokenizer, 1, prompt)
            phase_trial(torch, model, ids, 8)
            with torch.inference_mode(), profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                with record_function("topic02_prefill"):
                    out = model(ids, use_cache=True, logits_to_keep=1)
                    past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                    out = None
                with record_function("topic02_decode_7_steps"):
                    for _ in range(7):
                        out = model(token, past_key_values=past, use_cache=True, logits_to_keep=1)
                        past, token = out.past_key_values, out.logits[:, -1].argmax(-1, keepdim=True)
                        out = None
                torch.cuda.synchronize()
            prof.export_chrome_trace(str(output_dir / f"topic02-trace-p{prompt}.json"))
            events = sorted(prof.key_averages(), key=lambda e: e.self_device_time_total, reverse=True)[:12]
            summaries.append({"input_tokens": prompt, "output_tokens": 8, "model": list(MODELS)[0],
                              "operators": [{"name": e.key, "calls": e.count,
                                             "self_device_us": e.self_device_time_total} for e in events]})
            ids = out = past = token = prof = events = None
            gc.collect()
            resource_gate(torch, baseline)
            checkpoint(output_dir / "topic02-profile-summary.json", summaries)
    finally:
        model = tokenizer = ids = out = past = token = None
        gc.collect()
        torch.cuda.empty_cache()
    return summaries


## 2. Resolve pinned models once

Set **LSM_MODEL_CACHE** to persistent storage before running this cell (for example a mounted persistent volume). The default cache is persistent only while this runtime survives. An offline instructor run sets HF_HUB_OFFLINE=1; missing snapshots then fail without downloading. Never delete model caches when making a fresh evidence directory.


In [ ]:
from huggingface_hub import snapshot_download
cache = Path(os.environ.get('LSM_MODEL_CACHE', str(Path.home() / '.cache' / 'lsm-models')))
cache.mkdir(parents=True, exist_ok=True)
MODEL_PATHS = {}
for model_id, revision in MODELS.items():
    try:
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache, local_files_only=True)
    except FileNotFoundError:
        if os.environ.get('HF_HUB_OFFLINE') == '1':
            raise RuntimeError('Pinned model is missing from the offline cache')
        resolved = snapshot_download(model_id, revision=revision, cache_dir=cache)
    MODEL_PATHS[model_id] = resolved
print('Pinned snapshots resolved; local paths remain private.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EVIDENCE = Path('evidence') / RUN_ID


### Optional: execute that exact prompt locally, once

**Default: off. Not a required artifact or a substitute for the 50-trial matrix.** The recorded replay above always works without this cell. The embedded helper is the same canonical source used for the instructor demonstration and consumes the already resolved immutable 1.5B snapshot; it never downloads a model. It is deliberately limited to the instructor-style Linux/WSL safety envelope: at least **24 GiB available host RAM**, at most 12 GiB device use, one GPU process, one warmup request and one measured request, at most 24 generated IDs. Many Colab machines do not satisfy the host gate; do not lower it to force this optional demonstration.

If enabled, keep the notebook's existing safety guidance and your host's external process memory/time limits. The helper releases the model on exit and rechecks host/device headroom; this is not an assertion that allocation returned to its initial baseline. Stop on any failure instead of blindly retrying. Per-step synchronization, token extraction and text inspection alter the execution; this is an explanation of state, not an additional speed benchmark. Exact timing is not expected to reproduce the instructor's values, and implementation/runtime changes can change greedy output.

All required matrix cells below remain unchanged and use their own protocol, outputs and validator. Do not run live demo and matrix in parallel.


In [ ]:
RUN_CONCRETE_PROMPT_LIVE = False
if RUN_CONCRETE_PROMPT_LIVE:
    optional_dir = EVIDENCE / 'optional-concrete-prompt'
    optional_dir.mkdir(parents=True, exist_ok=False)  # reject an existing attempt before GPU work
    # Isolated namespace prevents the demo's constants from changing the required lab protocol.
    demo_namespace = {name: globals()[name] for name in ('GIB', 'MODELS', 'host_available_bytes', 'kv_bytes_per_token')}
    exec("\"\"\"One real request, recorded token by token. Not a benchmark or protocol-v2 trial.\n\nImport is CPU-only. Live execution requires Linux/WSL, an idle CUDA GPU, a pinned\nlocal snapshot and >=24 GiB available host RAM. No downloads are performed here.\n\"\"\"\nimport gc\nimport math\nimport time\nfrom pathlib import Path\n\n\nPROTOCOL = \"topic02-concrete-prompt-v1\"\nMODEL_ID = \"Qwen/Qwen2.5-1.5B-Instruct\"\nMODEL_REVISION = MODELS[MODEL_ID]\nMESSAGES = [\n    {\"role\": \"system\", \"content\": \"You are a helpful assistant. Reply with only the requested numbers.\"},\n    {\"role\": \"user\", \"content\": \"Continue the sequence with three numbers: 2, 4, 6,\"},\n]\nMAX_NEW_TOKENS = 24\n\n\ndef token_record(tokenizer, token_id, index=None):\n    value = {\"id\": int(token_id), \"piece\": tokenizer.convert_ids_to_tokens(int(token_id)),\n             \"text\": tokenizer.decode([int(token_id)], skip_special_tokens=False,\n                                      clean_up_tokenization_spaces=False)}\n    if index is not None:\n        value[\"index\"] = index\n    return value\n\n\ndef validate_replay(data):\n    \"\"\"CPU-only contract: visible selections lag the retained KV cache by one.\"\"\"\n    def positive_duration(value):\n        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:\n            raise ValueError(\"measured duration must be finite and positive\")\n\n    def byte_counter(value):\n        # JSON byte counts are integers. Reject bool as well as float NaN/Inf;\n        # relying on ordinary comparisons alone lets NaN peaks pass unnoticed.\n        if type(value) is not int or value < 0:\n            raise ValueError(\"memory counter must be a finite nonnegative integer\")\n\n    if data[\"protocol\"] != PROTOCOL:\n        raise ValueError(\"unexpected replay protocol\")\n    prompt = data[\"prompt\"]\n    p = prompt[\"input_token_count\"]\n    if p != len(prompt[\"input_token_ids\"]) or p < 1:\n        raise ValueError(\"prompt length mismatch\")\n    if [t[\"id\"] for t in prompt[\"input_tokens\"]] != prompt[\"input_token_ids\"]:\n        raise ValueError(\"prompt pieces do not match IDs\")\n    replay = data[\"replay\"]\n    positive_duration(replay[\"instrumented_request_wall_ms\"])\n    steps = replay[\"steps\"]\n    if not 1 <= len(steps) <= data[\"generation\"][\"max_new_tokens\"] <= MAX_NEW_TOKENS:\n        raise ValueError(\"output length exceeds the bounded contract\")\n    if [s[\"selected_token\"][\"id\"] for s in steps] != replay[\"output_token_ids\"]:\n        raise ValueError(\"selected IDs mismatch\")\n    previous_id = None\n    for index, step in enumerate(steps, 1):\n        expected_input = prompt[\"input_token_ids\"] if index == 1 else [previous_id]\n        if step[\"output_index\"] != index or step[\"input_token_ids\"] != expected_input:\n            raise ValueError(\"generation dependency mismatch\")\n        if step[\"input_shape\"] != [1, len(expected_input)]:\n            raise ValueError(\"input shape mismatch\")\n        if step[\"phase\"] != (\"prefill\" if index == 1 else \"decode\"):\n            raise ValueError(\"phase mismatch\")\n        if step[\"kv_tokens_before\"] != (0 if index == 1 else p + index - 2):\n            raise ValueError(\"cache before mismatch\")\n        if step[\"kv_tokens_after\"] != p + index - 1:\n            raise ValueError(\"cache after mismatch\")\n        for key in (\"wall_ms\", \"cuda_event_ms\"):\n            positive_duration(step[key])\n        mem = step[\"memory\"]\n        for key in (\"allocated_bytes\", \"reserved_bytes\", \"peak_allocated_bytes\", \"peak_reserved_bytes\"):\n            byte_counter(mem[key])\n        if not (0 <= mem[\"allocated_bytes\"] <= mem[\"reserved_bytes\"]):\n            raise ValueError(\"allocator nesting mismatch\")\n        if mem[\"peak_allocated_bytes\"] < mem[\"allocated_bytes\"] or mem[\"peak_reserved_bytes\"] < mem[\"reserved_bytes\"]:\n            raise ValueError(\"peak memory is below current memory\")\n        previous_id = step[\"selected_token\"][\"id\"]\n    if replay[\"final_kv_tokens\"] != p + len(steps) - 1:\n        raise ValueError(\"final KV lag mismatch\")\n    eos = data[\"generation\"][\"eos_token_ids\"]\n    if any(t in eos for t in replay[\"output_token_ids\"][:-1]):\n        raise ValueError(\"generation continued after EOS\")\n    if replay[\"stop_reason\"] == \"eos\":\n        if replay[\"output_token_ids\"][-1] not in eos:\n            raise ValueError(\"EOS stop without EOS\")\n    elif replay[\"stop_reason\"] != \"max_new_tokens\" or len(steps) != data[\"generation\"][\"max_new_tokens\"]:\n        raise ValueError(\"invalid stop reason\")\n    for sample in data[\"resource_samples\"]:\n        for key in (\"host_available_bytes\", \"device_used_bytes\", \"allocated_bytes\", \"reserved_bytes\",\n                    \"peak_allocated_bytes\", \"peak_reserved_bytes\"):\n            byte_counter(sample[key])\n    return True\n\n\ndef memory_sample(torch):\n    return {\"allocated_bytes\": torch.cuda.memory_allocated(),\n            \"reserved_bytes\": torch.cuda.memory_reserved(),\n            \"peak_allocated_bytes\": torch.cuda.max_memory_allocated(),\n            \"peak_reserved_bytes\": torch.cuda.max_memory_reserved()}\n\n\ndef resource_gate(torch):\n    available = host_available_bytes()\n    free, total = torch.cuda.mem_get_info()\n    used = total - free\n    if available < 24 * GIB or used > 12 * GIB or torch.cuda.max_memory_allocated() > 12 * GIB:\n        raise RuntimeError(\"STOP: require >=24 GiB available RAM and <=12 GiB device-used memory\")\n    return {\"host_available_bytes\": available, \"device_used_bytes\": used, **memory_sample(torch)}\n\n\ndef run_demo(model_path):\n    \"\"\"Warm one request, then record one instrumented replay; local snapshot only.\n\n    Each interval encloses model forward + argmax. Synchronization at every token,\n    CPU ID extraction, token decoding and resource checks perturb the request;\n    they make this explanation inspectable, not a valid service throughput test.\n    \"\"\"\n    import torch\n    import transformers\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    model_path = Path(model_path)\n    if model_path.name != MODEL_REVISION:\n        raise RuntimeError(\"Only the pinned model snapshot is supported\")\n    if transformers.__version__ != \"4.51.3\" or not torch.cuda.is_available():\n        raise RuntimeError(\"Require Transformers 4.51.3 and a CUDA GPU\")\n    data = {\"protocol\": PROTOCOL, \"scope\": \"one instrumented natural-EOS request; not a benchmark\",\n            \"model\": {\"id\": MODEL_ID, \"revision\": MODEL_REVISION},\n            \"environment\": {\"torch\": torch.__version__, \"transformers\": transformers.__version__,\n                            \"cuda\": torch.version.cuda, \"gpu\": torch.cuda.get_device_name(),\n                            \"dtype\": \"float16\", \"attention_requested\": \"sdpa\", \"logits_to_keep\": 1},\n            \"generation\": {\"max_new_tokens\": MAX_NEW_TOKENS, \"do_sample\": False, \"batch\": 1},\n            \"measurement\": {\"warmup_requests\": 1, \"measured_requests\": 1,\n                \"interval\": \"synchronized wall and CUDA events: forward + argmax; excludes CPU token extraction/text rendering/resource checks\",\n                \"memory_peak_scope\": \"reset before each individual forward+argmax; model remains loaded\",\n                \"limitations\": [\"single request, no distribution or performance guarantee\",\n                    \"per-token synchronization and inspection alter the execution\",\n                    \"not HTTP TTFT/TPOT, not summed CUDA kernel time, not protocol v2\"]},\n            \"resource_samples\": []}\n    model = tokenizer = ids = past = out = token = current = None\n    try:\n        data[\"resource_samples\"].append({\"stage\": \"before_load\", **resource_gate(torch)})\n        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)\n        model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True,\n            torch_dtype=torch.float16, attn_implementation=\"sdpa\").to(\"cuda\").eval()\n        chat = tokenizer.apply_chat_template(MESSAGES, tokenize=False, add_generation_prompt=True)\n        input_ids = tokenizer(chat, add_special_tokens=False)[\"input_ids\"]\n        ids = torch.tensor([input_ids], dtype=torch.long, device=\"cuda\")\n        p = len(input_ids)\n        eos = model.generation_config.eos_token_id\n        eos = [eos] if isinstance(eos, int) else list(eos)\n        data[\"generation\"][\"eos_token_ids\"] = eos\n        data[\"prompt\"] = {\"messages\": MESSAGES, \"rendered_chat\": chat,\n            \"input_token_ids\": input_ids, \"input_token_count\": p,\n            \"input_tokens\": [token_record(tokenizer, t, i) for i, t in enumerate(input_ids)]}\n        data[\"model\"].update(kv_bytes_per_token=kv_bytes_per_token(model.config),\n            layers=model.config.num_hidden_layers, kv_heads=model.config.num_key_value_heads,\n            attention_heads=model.config.num_attention_heads, hidden_size=model.config.hidden_size,\n            attention_selected=model.config._attn_implementation)\n        data[\"resource_samples\"].append({\"stage\": \"post_load\", **resource_gate(torch)})\n        # Same prompt and natural-EOS path, no timing/profiling retained as evidence.\n        with torch.inference_mode():\n            for index in range(MAX_NEW_TOKENS):\n                out = model(ids if index == 0 else token, past_key_values=past,\n                            use_cache=True, logits_to_keep=1)\n                token = out.logits[:, -1].argmax(-1, keepdim=True)\n                past = out.past_key_values\n                out = None\n                selected = int(token.item())\n                resource_gate(torch)\n                if selected in eos:\n                    break\n        token = past = None\n        gc.collect()\n        torch.cuda.synchronize()\n        # Keep the warmed allocator pool: never empty_cache between warmup and replay.\n        data[\"resource_samples\"].append({\"stage\": \"after_warmup\", **resource_gate(torch)})\n        generated, steps = [], []\n        replay_start = time.perf_counter()\n        with torch.inference_mode():\n            for index in range(MAX_NEW_TOKENS):\n                resource_gate(torch)\n                current = ids if index == 0 else token\n                before = 0 if past is None else int(past.get_seq_length())\n                torch.cuda.synchronize()\n                torch.cuda.reset_peak_memory_stats()\n                start = torch.cuda.Event(enable_timing=True)\n                end = torch.cuda.Event(enable_timing=True)\n                wall_start = time.perf_counter()\n                start.record()\n                out = model(current, past_key_values=past, use_cache=True, logits_to_keep=1)\n                token = out.logits[:, -1].argmax(-1, keepdim=True)\n                past = out.past_key_values\n                out = None\n                end.record()\n                end.synchronize()\n                wall_ms = (time.perf_counter() - wall_start) * 1000\n                memory = memory_sample(torch)\n                selected = int(token.item())\n                generated.append(selected)\n                steps.append({\"output_index\": index + 1, \"phase\": \"prefill\" if index == 0 else \"decode\",\n                    \"input_shape\": list(current.shape),\n                    \"input_token_ids\": input_ids if index == 0 else [generated[-2]],\n                    \"kv_tokens_before\": before, \"kv_tokens_after\": int(past.get_seq_length()),\n                    \"selected_token\": token_record(tokenizer, selected),\n                    \"decoded_prefix\": tokenizer.decode(generated, skip_special_tokens=False,\n                                                        clean_up_tokenization_spaces=False),\n                    \"wall_ms\": wall_ms, \"cuda_event_ms\": start.elapsed_time(end), \"memory\": memory})\n                resource_gate(torch)\n                if selected in eos:\n                    break\n        data[\"replay\"] = {\"output_token_ids\": generated,\n            \"output_text\": tokenizer.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False),\n            \"output_text_with_special_tokens\": tokenizer.decode(generated, skip_special_tokens=False,\n                                                                clean_up_tokenization_spaces=False),\n            \"stop_reason\": \"eos\" if generated[-1] in eos else \"max_new_tokens\",\n            \"steps\": steps, \"final_kv_tokens\": int(past.get_seq_length()),\n            \"instrumented_request_wall_ms\": (time.perf_counter() - replay_start) * 1000}\n        validate_replay(data)\n        data[\"resource_samples\"].append({\"stage\": \"after_replay\", **resource_gate(torch)})\n    finally:\n        model = tokenizer = ids = past = out = token = current = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        torch.cuda.synchronize()\n        data[\"resource_samples\"].append({\"stage\": \"after_release\", **resource_gate(torch)})\n    return data\n", demo_namespace)
    LIVE_PROMPT = demo_namespace['run_demo'](MODEL_PATHS['Qwen/Qwen2.5-1.5B-Instruct'])
    checkpoint(optional_dir / 'replay.json', LIVE_PROMPT)
    print('Actual messages:', LIVE_PROMPT['prompt']['messages'])
    print('Actual output:', repr(LIVE_PROMPT['replay']['output_text']))
    print('Stop reason:', LIVE_PROMPT['replay']['stop_reason'])
else:
    print('Optional live demonstration skipped; recorded replay remains available without a GPU.')


## 3. Startup + smoke — stop here if it fails

One small-model 128/32 case, one measured trial after warmup. Inspect memory recovery and the timings before continuing. A successful smoke is **not** the completed lab. If interrupted, retain its JSON and restart only after understanding the failure.


In [ ]:
SMOKE = run_matrix(MODEL_PATHS, EVIDENCE / 'smoke', smoke=True)
print(json.dumps(summarize(SMOKE), indent=2))
assert SMOKE['status'] == 'complete'


## 4. Required matrix + matched batch comparison

Expect 50 raw trials, checkpointed after each trial. A new attempt needs a new RUN_ID; existing results cannot be overwritten. Do not change shape, dtype, attention or counts mid-run. The two models load sequentially and are released even on an exception. The matrix is deliberately small; it does not establish production capacity.


In [ ]:
assert SMOKE['status'] == 'complete', 'Pass the smoke first'
RUN = run_matrix(MODEL_PATHS, EVIDENCE / 'core')
SUMMARY = summarize(RUN)
checkpoint(EVIDENCE / 'topic02-summary.json', SUMMARY)
for row in SUMMARY:
    print(json.dumps(row))


## 5. CPU artifact contract

This checker rejects incomplete matrices, duplicated trials, wrong revisions, nonfinite timings and off-by-one/rate errors. Passing validates structure and arithmetic, **not authenticity or the correctness of your bottleneck conclusion**. It is also available in the public template as scripts/validate_topic02.py.


In [ ]:
#!/usr/bin/env python3
"""CPU-only structure/arithmetic check, not a proof that a benchmark is genuine."""
import argparse
import json
import math
from pathlib import Path

MODELS = {
    "Qwen/Qwen2.5-0.5B-Instruct": "7ae557604adf67be50417f59c2c2f167def9a775",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CASES = [(1, 128, 32), (1, 4096, 32), (1, 128, 128), (1, 4096, 128), (4, 128, 32)]


def validate(data):
    if (data.get("protocol"), data.get("status"), data.get("scope"), data.get("trials")) != (
            "topic02-phase-v2", "complete", "core", 5):
        raise ValueError("Expected a completed core protocol-v2 run with five trials; smoke/partial is not complete")
    if data.get("models") != MODELS or sorted(data.get("cases", [])) != sorted(map(list, CASES)):
        raise ValueError("Pinned model revisions or required cases differ")
    env = data.get("environment", {})
    if any(env.get(k) != v for k, v in {"dtype": "float16", "transformers": "4.51.3",
                                        "attention_requested": "sdpa", "logits_to_keep": 1}.items()):
        raise ValueError("Protocol runtime settings differ")
    if not all(env.get(k) for k in ("python", "torch", "cuda", "gpu")):
        raise ValueError("Missing environment provenance")
    expected = {(model, *case, repeat) for model in MODELS for case in CASES for repeat in range(5)}
    seen = set()
    for row in data.get("results", []):
        key = tuple(row[k] for k in ("model", "batch", "input_tokens", "output_tokens", "repeat"))
        if key not in expected or key in seen:
            raise ValueError("Unexpected or duplicate model/shape/trial")
        seen.add(key)
        model, batch, prompt, output, repeat = key
        for metric in ("prefill_wall_ms", "prefill_event_ms", "decode_wall_ms", "decode_event_ms",
                       "decode_aggregate_tokens_per_s", "decode_mean_step_ms", "peak_allocated_bytes",
                       "peak_reserved_bytes", "prefill_peak_allocated_bytes"):
            value = row.get(metric)
            if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or value <= 0:
                raise ValueError(f"{metric} must be a finite positive measurement")
        if (row.get("decode_steps_per_sequence"), row.get("generated_tokens"),
            row.get("retained_tokens_per_sequence")) != (output - 1, batch * output, prompt + output - 1):
            raise ValueError("Output/step/KV accounting mismatch")
        if not math.isclose(row["decode_aggregate_tokens_per_s"], batch * (output - 1) * 1000 / row["decode_wall_ms"], rel_tol=1e-7):
            raise ValueError("Aggregate decode rate uses the wrong numerator or clock")
        if not math.isclose(row["decode_mean_step_ms"], row["decode_wall_ms"] / (output - 1), rel_tol=1e-7):
            raise ValueError("Mean step interval uses the wrong denominator")
        if row["peak_reserved_bytes"] < row["peak_allocated_bytes"] or row["peak_allocated_bytes"] < row["prefill_peak_allocated_bytes"]:
            raise ValueError("Impossible allocated/reserved peak ordering")
    if seen != expected:
        raise ValueError(f"Missing trials: found {len(seen)} of {len(expected)}")
    return len(seen)



print('Validated raw trials:', validate(RUN))


## 6. Focused profiles — two shapes, not the entire matrix

The 0.5B model, P=128 and P=4096, B=1, O=8. Each trace has named prefill and seven-step decode ranges. Open the local Chrome traces in a trace viewer; do not upload private artifacts to a public service. Record the visible gaps and dominant operators separately for each phase. Requested SDPA is not proof of a particular FlashAttention kernel: identify the actual selected operator/kernel from the trace.

Profiling changes execution. Keep these traces separate from unprofiled timings. They cannot explain the 1.5B/128-output case directly, establish HBM traffic, or validate a production p95. Hardware-counter diagnosis with Nsight is guided self-study, not required for this core artifact.


In [ ]:
PROFILE = focused_profiles(MODEL_PATHS[list(MODELS)[0]], EVIDENCE / 'profiles')
print(json.dumps(PROFILE, indent=2))


## 7. Required analysis and hand-in

Keep this lab in your **private** course repository; follow the current round's tag/deadline procedure in the student guide. Do not change the leaderboard server contract just to run a notebook.

Submit **topic02-analysis.md**, the unmodified core **topic02-benchmark.json**, **topic02-summary.json**, both **topic02-trace-p128.json / topic02-trace-p4096.json**, and **topic02-profile-summary.json**. No model weights, caches, secrets or executed notebook outputs belong in the public template.

Your analysis must contain:

1. A manifest: model revisions, GPU, driver, runtime, dtype, attention path, exact cases and timing boundaries. Add the driver from nvidia-smi to your report; the notebook records the other runtime fields.
2. Three predictions written before the run; observed medians **and min/max plus all five raw trials**. Five samples are not credible p95.
3. Two matched input/output contrasts for each model and the B1→B4 128/32 comparison. Aggregate rate uses B×(O−1); per-sequence mean cadence uses decode wall time/(O−1). Do not equate either with service capacity.
4. A KV calculation from model.config and P+O−1 retained tokens, reconciled with allocated/reserved/peak bytes. Explain at least two omitted memory terms.
5. A roofline estimate with declared precision, memory boundary, FLOP convention and **source for machine bandwidth/compute**. A worked hypothetical roof is allowed if labeled; do not fabricate hardware counters.
6. Two trace observations (phase + prompt length + operator/range), an alternative explanation, one failed or unresolved prediction, and the smallest next discriminating test. A screenshot without interpretation is insufficient.
7. Safety/reproducibility: partial failures if any, resource peaks/recovery, cache reuse and exact run identifier. Never hide an OOM or silently shrink the required matrix.

**Success criterion:** a reviewer can reproduce your arithmetic and understand why the evidence supports—or fails to support—your hypothesis. A faster number alone is not the learning outcome. Optional extensions must be labeled separately and must retain their own raw evidence.
